# Landsat vegetation and water indices over the Ogallala aquifer at 30 m: monthly medians on tiles, one CF file per tile and year

Landsat 5, 7, 8 and 9 (Collection 2 Level-2 surface reflectance, Tier 1), for the months you choose
(2005 → now by default), at **Landsat's own 30 m pixels**. The pixels are downloaded to Colab's scratch
disk as the stored integers and everything is computed here, using eemont's index formulas, so almost
none of the project's Earth Engine compute quota is used.

| | |
|---|---|
| Output | `MyDrive/MSUGWB/LANDSAT_C2L2_indices_30m_monthly_tiles_v2/<year>/<tile>/LANDSAT_C2L2_indices_30m_monthly_tiles_v2_<tile>_<year>_Ogallala.nc`: a folder per year, in it a folder per tile, in it that tile's file of the year |
| Months | `START_MONTH` to `STOP_MONTH` (Cell 2), e.g. `"2005-01"` to `"2005-12"` |
| One record | one month: per pixel, the median of all its usable observations of that month, from every path/row that sees it |
| Grid | Landsat's own 30 m grid of each UTM zone, cut into tiles of 5120 × 5120 pixels (153.6 km). Scenes are placed on it by whole-pixel shifts; **nothing is resampled** |
| Pixels kept | pixels whose centre lies inside the aquifer polygon |
| Variables | 18 indices as 16-bit integers with a scale factor; `n_obs`; `n_obs_oli` (how many of them are Landsat 8/9); `obs_day` (their mean day of the month); `NDVI_range`, `NDMI_range` (spread within the month); per month the scenes used |

**Indices** (set in Cell 2): NDVI, EVI, EVI2, SAVI, MSAVI, OSAVI, GNDVI, NIRv, kNDVI · NDMI, NMDI, MSI, GVMI ·
NDWI, MNDWI, AWEInsh, AWEIsh, WI2015.

**How to run**

1. Runtime → Change runtime type → a **GPU** (T4 is enough). A CPU runtime works but the maths is slow at 30 m.
2. *Run all* with `TEST_MODE = True`: 2 tiles, June and July of 2022, into `.../_test/`, every check
   switched on, and a projection of runtime, download volume and Drive size for the months from
   `START_MONTH` to `STOP_MONTH`.
3. Set `TEST_MODE = False`, set `START_MONTH` and `STOP_MONTH`, and *Run all*. Whenever the runtime dies or
   you stop it, run all again. `ONLY_TILES` lets several Colab sessions work on different tiles; never
   run two sessions on the same tile.

**Choosing the months.** `START_MONTH = "2005-01"`, `STOP_MONTH = "2005-12"` makes and exports those twelve
months and nothing else; `STOP_MONTH = None` goes on to the latest month that is ready. Ranges can be run
in any order and may overlap. A later run adds its months to the yearly files that are on Drive; a month
that is already there is never computed again, and its stored values are never changed.

**What makes the values analysis ready**

* **Screening.** An observation is used unless it is fill, dilated cloud, cirrus, cloud, cloud shadow or
  snow (`QA_PIXEL`), saturated (`QA_RADSAT`), or has a reflective band outside USGS's valid range
  7273–43636. Water is kept: three of the indices are for open water.
* **Haze.** With `AEROSOL_SCREEN` an observation is also left out when the air was hazy: aerosol level
  "high" in `SR_QA_AEROSOL` (Landsat 8/9), or atmospheric opacity above 0.3 in `SR_ATMOS_OPACITY` (Landsat 5/7).
* **Landsat 7 after 2016.** Its orbit has drifted to earlier overpass times since 2017, and USGS warns that
  this can make its data unreliable for quantitative work. `LANDSAT7_LAST_DAY` leaves those scenes out.
* **What each value rests on.** `obs_day` tells when in the month the observations fell, `n_obs_oli` which
  sensor family they came from, and `<index>_range` how much they differed.
* **Reflectance.** Stored integers become surface reflectance with the scale and offset eemont holds for
  each band; every index is then computed per pixel and per scene.
* **Monthly median from every pass.** For each pixel and index, the median of the month's usable
  observations (the mean of the two middle ones when their number is even). Where neighbouring paths
  overlap, a pixel gets the observations of both, so there is one value per pixel and no line where one
  path's scenes end. One observation is one satellite pass: two scenes of the same pass that overlap
  count once. `n_obs` says how many observations there were; 0 means none.
* **One grid per UTM zone.** Every Landsat scene of a UTM zone is on the same 30 m lattice, whatever its
  path/row. Tiles are pieces of that lattice, so the tiles of a zone fit together exactly and every
  yearly file of a tile has the same `x` and `y`.
* **Scene edges.** No extra trimming is applied, because none was found to be needed: in three test
  scenes (Landsat 5, 7, 8) the disturbed pixels along scene edges and gap stripes were already removed
  by the screening above, and the pixels left there differed from the interior by less than 0.003 in NDVI.
* **Compact, self-describing values.** 16-bit integers with a `scale_factor` (0.0001 for most indices), so
  xarray and GDAL return ordinary numbers and `NaN` for missing ones.

**Sensor harmonisation is a switch, on by default (`HARMONIZE`).** Landsat 5/7 (TM/ETM+) and Landsat 8/9
(OLI) bands differ slightly, so their indices do too. With `HARMONIZE = True` the TM/ETM+ reflectance is
adjusted to OLI before the indices are computed, with the coefficients of Roy et al. (2016, Table 2, OLS).
Read this before switching it on: those coefficients were fitted to data from before Landsat's
collections, and Google's Earth Engine guide now calls the procedure outdated and "not recommended" for
Collection 2 surface reflectance; nobody has published the same analysis for Collection 2. Whichever
you choose is written into every file (`sensor_harmonisation`), and files of the two kinds cannot be mixed
in one output folder.

**How it works** (per tile and year: fetch a month → check → compose → next month → check the file → Drive)

* The scene list comes from Earth Engine's catalogue and is cached on Drive. Each tile's frame (the tile
  cut to the aquifer) is stored in `frames.json` and never changed afterwards.
* For a tile and month, every scene of the tile's UTM zone that reaches it is used, whatever its
  path/row. Only the part of a scene that lies on the tile is fetched.
* A scene is fetched in blocks as the stored 16-bit integers, 8 bands per request, up to 32 requests at
  a time. Earth Engine refuses a request above 48 MiB, counting 3 bytes per band value, so a block holds
  at most 1448 × 1448 pixels (1365 × 1365 with the aerosol band as a ninth); a refused request is fetched in halves instead of failing. Every block is
  checked when it arrives and compared with a checksum when it is read back from scratch.
* While one month is being computed and written, the next one is already downloading.
* A month is ready once it ended 45 days ago, so that its Tier-1 scenes have arrived; months of the range
  that are not ready yet are left for a later run. A yearly file holds the months made so far. A later
  run that wants more months of that year fetches the file from Drive and adds them. If a new month
  belongs before one that is there, a new file is written and the months on Drive are copied into it as
  they are stored.

**Stopping and resuming**

* The unit of work is one tile and one year: a few minutes. Its file is built on scratch, read back
  and audited, and only then copied to Drive under its final name, with a checksum comparison. Nothing
  on Drive is ever half written.
* A runtime that dies loses the file it was working on and nothing else. The next run starts with the
  first file that is missing or lacks a month of the range.
* Files that an earlier version of this notebook stored as `<tile>/<file>` are moved to `<year>/<tile>/<file>`
  at the start of a run (renamed on Drive, not copied or changed).
* If a file on Drive cannot be read, the run stops and changes nothing. It never replaces a file on its own.

**What remains between tiles and zones**

* Inside a UTM zone the tiles join without a seam in the grid. The values still change gradually across
  the band where two paths overlap, because a pixel outside that band is seen by one path only and so on
  other days than its neighbours. Monthly Landsat data cannot avoid this; `n_obs` shows where it happens.
* The aquifer lies in UTM zones 13 and 14 (and a corner of 15). Scenes of different zones are on different
  grids and are **not** merged, since that would need resampling. Along the zone boundary both zones hold
  the same ground, each from its own scenes; a single aquifer-wide grid needs one of them reprojected.

**What is not done.** No gap filling (Landsat 7 stripes after May 2003 mean fewer observations, not
invented ones), no view-angle (BRDF) or terrain correction. EVI, NMDI and MSI can take extreme values
where a denominator passes through zero; a monthly value too large for its scale is stored as missing.

In [ ]:
# ═══════════════ CELL 1: install ═══════════════
# earthengine-api, numpy, cupy (on GPU runtimes), geopandas, shapely and pyproj are part of Colab.
# If Colab offers "RESTART RUNTIME" after this cell, restart, then Run all again.
!pip install -q "eemont==2025.7.1" netCDF4 rasterio 2>&1 | tail -2

In [ ]:
# ═══════════════ CELL 2: mount Drive, stage the shapefile, SETTINGS ═══════════════
import os, zipfile
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

SHP_ZIP  = '/content/drive/MyDrive/high_plains_quifer.zip'
SHP_DIR  = '/content/ogallala_shp'
SHP_PATH = f'{SHP_DIR}/high_plains_quifer/hp_bound2010.shp'
if not os.path.exists(SHP_PATH):
    with zipfile.ZipFile(SHP_ZIP) as z:
        z.extractall(SHP_DIR)
print("Shapefile ready:", os.path.exists(SHP_PATH))

# ================= SETTINGS (edit these, then Run all) =================
TEST_MODE     = True       # True -> short rehearsal into .../_test/ ; False -> the months from START_MONTH to STOP_MONTH
TEST_TILES    = 2          # tiles in the rehearsal (one mostly inside the aquifer, one on its edge)
TEST_YEAR     = 2022       # year of the rehearsal
TEST_MONTHS   = [6, 7]     # months of that year in the rehearsal
TEST_FRAME_PX = None       # None -> whole tiles; or e.g. 2048 to rehearse on a 2048 x 2048 pixel square of each

# ---- which months are processed and exported (not used by the rehearsal) ----
START_MONTH = "2005-01"    # first month, written "YYYY-MM"
STOP_MONTH  = None         # last month, written "YYYY-MM" (it is included); None -> the latest month that is ready.
                           # Any range can be run in any order: a later run with another range adds its months
                           # to the yearly files that are already on Drive and never computes a month twice

PROJECT_ID = "msugw-503806"                                    # Earth Engine cloud project
EE_URL     = "https://earthengine-highvolume.googleapis.com"   # endpoint meant for many parallel requests
SATELLITES = ["LT05", "LE07", "LC08", "LC09"]                  # Landsat 5, 7, 8, 9 (Collection 2 Level-2)
TIER       = "T1"                                              # Tier 1 only (USGS's time-series quality)
SETTLE_DAYS = 45           # a month is ready once it ended this many days ago, so that its Tier-1
                           # scenes have arrived (USGS needs up to about a month to finish them)
LANDSAT7_LAST_DAY = "2016-12-31"   # Landsat 7 scenes after this day are left out: its orbit has drifted to earlier
                           # overpass times since 2017 (USGS). None -> every Landsat 7 scene is used
TILE_PX    = 5120          # edge of an output tile in 30 m pixels (153.6 km). Do not change it once files exist
ONLY_TILES = None          # None -> every tile; or e.g. ["z14e02n29", "z14e02n28"] (to split the work over sessions)
SKIP_SCENES   = []         # scene ids to treat as permanently unavailable (see the stall message)

# ---- indices (formulas, names and references come from eemont's catalogue) ----
INDICES = ["NDVI", "EVI", "EVI2", "SAVI", "MSAVI", "OSAVI", "GNDVI", "NIRv", "kNDVI",     # greenness
           "NDMI", "NMDI", "MSI", "GVMI",                                                  # canopy water
           "NDWI", "MNDWI", "AWEInsh", "AWEIsh", "WI2015"]                                 # open water
INDEX_PARAMS     = dict(g=2.5, C1=6.0, C2=7.5, L=1.0)   # eemont's defaults (EVI, EVI2)
INDEX_PARAMS_FOR = {"SAVI": dict(L=0.5)}                # SAVI as published; eemont's shared default L=1 is EVI's
AWEINSH_AS_PUBLISHED = True   # True -> 4*(G-S1) - (0.25*N + 2.75*S2) as in Feyisa et al. 2014;
                              # False -> eemont's catalogue formula, which has "+ 2.75*S2"
# Values are stored as 16-bit integers: index = stored integer x scale. A scale of 0.0001 holds -3.2767 .. 3.2767.
INDEX_SCALE = dict(NDVI=1e-4, EVI=1e-4, EVI2=1e-4, SAVI=1e-4, MSAVI=1e-4, OSAVI=1e-4, GNDVI=1e-4, NIRv=1e-4,
                   kNDVI=1e-4, NDMI=1e-4, NMDI=1e-4, GVMI=1e-4, NDWI=1e-4, MNDWI=1e-4,
                   AWEIsh=2e-4, AWEInsh=2.5e-4,      # reach about +-3.5 and -7 .. 4
                   MSI=1e-3,                         # a ratio: 0 .. 32.767 is kept
                   WI2015=1e-2)                      # reaches about +-185

# ---- which 30 m observations enter a monthly value ----
QA_PIXEL_REJECT  = 0b111110   # QA_PIXEL bits 1-5: dilated cloud, cirrus, cloud, cloud shadow, snow
REJECT_SATURATED = True       # leave out pixels with any QA_RADSAT bit set
REQUIRE_VALID_RANGE = True    # leave out pixels with a reflective band outside USGS's valid range 7273..43636
AEROSOL_SCREEN = True         # leave out hazy observations: Landsat 8/9 pixels whose aerosol level is "high"
                              # (SR_QA_AEROSOL), Landsat 5/7 pixels whose atmospheric opacity is above OPACITY_MAX
OPACITY_MAX = 0.3             # SR_ATMOS_OPACITY of Landsat 5/7 (USGS: below 0.1 clear, 0.1-0.3 average, above 0.3 hazy)
HARMONIZE = True              # True -> Landsat 5/7 reflectance is adjusted to Landsat 8/9 with the coefficients
                              # of Roy et al. (2016) before the indices are computed. Read the note in the
                              # first cell: those coefficients predate Collection 2.
SPREAD_INDICES = ["NDVI", "NDMI"]   # for these, the range (largest minus smallest) of the month's observations
                                    # is stored as well, as <index>_range
USE_GPU = "auto"              # "auto" -> CuPy on the GPU when the runtime has one, otherwise NumPy on the CPU

# ---- download and compute ----
BLOCK_PX           = 1448  # a download block holds at most BLOCK_PX x BLOCK_PX pixels. Earth Engine counts 3 bytes
                           # per band value and refuses a request above 48 MiB, which for 8 bands is 1448 x 1448 (9 bands: 1365 x 1365).
                           # A larger value here is cut back to that limit by itself
N_DOWNLOAD_THREADS = 32    # blocks downloaded at a time (Earth Engine allows 40 concurrent requests)
REQUEST_DEADLINE_S = 180   # give up on a single request after this long (it is then retried)
STRIP_ROWS         = 512   # rows composed and written at a time (also the chunk height of the files)

COMPRESS_LEVEL = 2         # zlib level of the .nc files (lossless)
CHUNK_YX       = 512       # HDF5 chunk edge in y and x

MIN_FREE_GB       = 20.0   # scratch space that must stay free at all times
DISK_MAX_WAIT_S   = 2 * 3600
DRIVE_READ_TRIES  = 4      # attempts at reading a file's state from Drive before the run stops. The run never
                           # redoes or replaces a file because Drive failed to open it
MAX_RETRIES       = 4      # extra attempts per block
STALL_MINUTES     = 20     # stop cleanly after this long with no progress at all
READBACK_MONTHS   = 1      # months per file re-read from the closed file and compared with what was computed
                           # (every month in TEST_MODE)
EEMONT_CHECK_PX   = 48     # edge of the window compared pixel by pixel with eemont on Earth Engine

MANIFEST_MAX_AGE_DAYS = 7  # refresh the cached scene list when older than this
REBUILD_MANIFEST      = False
# =======================================================================

In [ ]:
# ═══════════════ CELL 3: imports, logging, paths, Earth Engine sign-in, eemont definitions, engine ═══════════════
import re, gc, io, ast, sys, json, gzip, glob, time, math, zlib, shutil, hashlib, logging, calendar, threading, warnings
from datetime import datetime, timezone, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import netCDF4 as nc4
import geopandas as gpd
import pyproj
import shapely
import rasterio.features
from affine import Affine
import ee
import eemont                      # index catalogue, scale/offset tables, and the server-side reference

# force=True: Colab installs its own root log handler, so a plain basicConfig() is a no-op there.
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s",
                    datefmt="%H:%M:%S", force=True)
log = logging.getLogger("landsat30")
warnings.filterwarnings("ignore", category=UserWarning, module="ee_extra")
for _n in ("urllib3.connectionpool", "googleapiclient", "google_auth_httplib2", "rasterio", "pyogrio"):
    logging.getLogger(_n).setLevel(logging.ERROR)

# ---------- names and paths (Drive = durable, scratch = fast and disposable) ----------
ROI_LABEL   = "Ogallala"
PRODUCT_TAG = "LANDSAT_C2L2_indices_30m_monthly_tiles_v2"
DRIVE_ROOT  = "/content/drive/MyDrive/MSUGWB"
DRIVE_OUT   = f"{DRIVE_ROOT}/{PRODUCT_TAG}" + ("/_test" if TEST_MODE else "")   # the rehearsal never touches real outputs
SCRATCH     = "/content/landsat30_scratch" + ("_test" if TEST_MODE else "")
RAW_DIR     = f"{SCRATCH}/raw"                      # downloaded 30 m pixels of one or two months, then deleted
NC_DIR      = f"{SCRATCH}/nc"                       # the yearly file being written
MASK_DIR    = f"{SCRATCH}/masks"                    # aquifer mask of each tile
DISK_PATH   = "/content"                            # same filesystem as scratch AND Drive's local upload cache

MANIFEST_NAME = "manifest_landsat_c2l2.json.gz"
FRAMES_NAME   = "frames.json"                       # the fixed 30 m frame of every tile (made once, never changed)
LEDGER_NAME   = "files_done.jsonl"                  # one line per yearly file that is on Drive
LOG_FAILED    = "failed_scenes.jsonl"
LOG_SKIPPED   = "skipped_scenes.jsonl"
for _d in (DRIVE_OUT, RAW_DIR, NC_DIR, MASK_DIR):
    os.makedirs(_d, exist_ok=True)

PIXEL = 30.0                                        # Landsat's own pixel size (m); nothing is resampled
TIME_UNITS, TIME_CALENDAR = "days since 1970-01-01 00:00:00", "proleptic_gregorian"
EPOCH = datetime(1970, 1, 1)

SAT_NAME   = {"LT04": "Landsat 4 TM", "LT05": "Landsat 5 TM", "LE07": "Landsat 7 ETM+",
              "LC08": "Landsat 8 OLI", "LC09": "Landsat 9 OLI-2"}
SAT_NUMBER = {"LT04": 4, "LT05": 5, "LE07": 7, "LC08": 8, "LC09": 9}
SAT_DOI    = {"LT04": "10.5066/P9IAXOVV", "LT05": "10.5066/P9IAXOVV", "LE07": "10.5066/P9C7I13B",
              "LC08": "10.5066/P9OGBGM6", "LC09": "10.5066/P9OGBGM6"}
# Band letters of the index catalogue -> Landsat Collection 2 Level-2 band names. Checked against
# eemont's own server-side result in eemont_cross_check() (Cell 7): a wrong letter fails that check.
BAND_LETTERS = {"OLI": dict(B="SR_B2", G="SR_B3", R="SR_B4", N="SR_B5", S1="SR_B6", S2="SR_B7"),
                "TM":  dict(B="SR_B1", G="SR_B2", R="SR_B3", N="SR_B4", S1="SR_B5", S2="SR_B7")}
SENSOR = {"LT04": "TM", "LT05": "TM", "LE07": "TM", "LC08": "OLI", "LC09": "OLI"}
LETTERS = ("B", "G", "R", "N", "S1", "S2")
SR_VALID = (7273, 43636)                             # USGS valid range of the reflective bands (DN)
QA_BANDS = ["QA_PIXEL", "QA_RADSAT"]
# The band that says how hazy the air was, and how it is stored (checked against real scenes):
# Landsat 8/9: SR_QA_AEROSOL, 8-bit, bits 6-7 = aerosol level (3 = high). Landsat 5/7: SR_ATMOS_OPACITY,
# signed 16-bit, opacity x 1000, -9999 where it was not retrieved.
AEROSOL_BAND = {"OLI": ("SR_QA_AEROSOL", "|u1"), "TM": ("SR_ATMOS_OPACITY", "<i2")}
RANGE_VARS = [f"{name}_range" for name in SPREAD_INDICES]
COUNT_VARS = ["n_obs", "n_obs_oli", "obs_day"]        # 8-bit variables written for every month
# Roy et al. (2016), Table 2, ordinary least squares, surface reflectance: OLI = intercept + slope x ETM+.
# Used only when HARMONIZE is True, and then for Landsat 4/5 TM as well as Landsat 7 ETM+.
HARMONIZATION = dict(B=(0.8474, 0.0003), G=(0.8483, 0.0088), R=(0.9047, 0.0061),
                     N=(0.8462, 0.0412), S1=(0.8937, 0.0254), S2=(0.9071, 0.0172))      # letter -> (slope, intercept)


class PipelineStop(RuntimeError):
    """Clean stop: everything on Drive is safe, re-running resumes."""


class DataIntegrityError(RuntimeError):
    """Something that must never reach the output was detected."""


class SceneUnavailable(Exception):
    """Earth Engine says the scene does not exist. Nothing else may be mistaken for this: a scene
    reported unavailable is left out of the record for good."""


class RunStopping(Exception):
    """Raised inside a download thread when the run is being stopped."""


class RequestTooLarge(Exception):
    """Earth Engine refused a pixel request for its size. Trying again cannot help; the rectangle
    is fetched as two halves instead."""


def utcnow():
    return datetime.now(timezone.utc).replace(tzinfo=None)   # naive UTC everywhere


def iso(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"


def parse_iso(s):
    s = s.rstrip("Z")
    return datetime.strptime(s, "%Y-%m-%dT%H:%M:%S.%f" if "." in s else "%Y-%m-%dT%H:%M:%S")


# ---------- Earth Engine ----------
def ee_login():
    """Stored credentials first (an unattended restart needs no clicks); otherwise the sign-in flow."""
    try:
        ee.Initialize(project=PROJECT_ID, opt_url=EE_URL)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=PROJECT_ID, opt_url=EE_URL)
    ee.data.setDeadline(REQUEST_DEADLINE_S * 1000)
    if ee.Number(1).add(1).getInfo() != 2:
        raise PipelineStop("Earth Engine did not answer a trivial request.")
    log.info("Earth Engine ready (project %s, high-volume endpoint)", PROJECT_ID)


def collection_id(sat):
    return f"LANDSAT/{sat}/C02/{TIER}_L2"


# ---------- index definitions, taken from eemont ----------
CATALOGUE = eemont.indices()                # eemont's bundled Awesome Spectral Indices catalogue
_KERNEL_RE = re.compile(r"^k(S1|S2|[BGRN])(S1|S2|[BGRN])$")
_AST_OK = (ast.Expression, ast.BinOp, ast.UnaryOp, ast.Constant, ast.Name, ast.Load,
           ast.Add, ast.Sub, ast.Mult, ast.Div, ast.Pow, ast.USub, ast.UAdd)


def _compile_formula(formula):
    """Catalogue formulas are plain arithmetic. Anything else is refused before it is evaluated."""
    tree = ast.parse(formula, mode="eval")
    names = set()
    for node in ast.walk(tree):
        if not isinstance(node, _AST_OK):
            raise ValueError(f"formula {formula!r} contains {type(node).__name__}")
        if isinstance(node, ast.Constant) and not isinstance(node.value, (int, float)):
            raise ValueError(f"formula {formula!r} contains a non-numeric constant")
        if isinstance(node, ast.Name):
            names.add(node.id)
    return compile(tree, "<index formula>", "eval"), names


def build_index_defs():
    defs = {}
    for name in INDICES:
        if name not in CATALOGUE:
            raise PipelineStop(f"'{name}' is not in eemont's index catalogue (see eemont.listIndices()).")
        if name not in INDEX_SCALE:
            raise PipelineStop(f"'{name}' has no entry in INDEX_SCALE (Cell 2): its storage step is not defined.")
        entry = CATALOGUE[name]
        formula, origin = str(entry["formula"]), f"eemont {eemont.__version__} catalogue"
        if name == "AWEInsh" and AWEINSH_AS_PUBLISHED:
            formula = "4.0 * (G - S1) - (0.25 * N + 2.75 * S2)"
            origin = ("Feyisa et al. 2014 as published; eemont's catalogue formula reads "
                      f"'{entry['formula']}' (opposite sign on the SWIR2 term)")
        code, names = _compile_formula(formula)
        params = dict(INDEX_PARAMS, **INDEX_PARAMS_FOR.get(name, {}))
        kernels, used = {}, {}
        for nm in sorted(names):
            if nm in LETTERS:
                continue
            m = _KERNEL_RE.match(nm)
            if m:
                kernels[nm] = (m.group(1), m.group(2))
            elif nm in params:
                used[nm] = float(params[nm])
            else:
                raise PipelineStop(f"{name}: '{nm}' in '{formula}' is neither a Landsat band letter nor a "
                                   "parameter in INDEX_PARAMS. Landsat cannot supply it.")
        letters = sorted({n for n in names if n in LETTERS} | {x for ab in kernels.values() for x in ab})
        defs[name] = dict(code=code, formula=formula, origin=origin, names=sorted(names), params=used,
                          kernels=kernels, letters=letters, long_name=str(entry["long_name"]),
                          reference=str(entry["reference"]), domain=str(entry["application_domain"]),
                          catalogue_formula=str(entry["formula"]), scale=float(INDEX_SCALE[name]))
    return defs


INDEX_DEFS = build_index_defs()


def eval_index(d, bands, xp):
    """One index from reflectance arrays keyed by band letter. Works on NumPy and CuPy arrays alike and
    keeps their precision. Kernel terms follow eemont's default: RBF with sigma = 0.5 * (a + b)."""
    ns = {}
    for nm in d["names"]:
        if nm in bands:
            ns[nm] = bands[nm]
        elif nm in d["params"]:
            ns[nm] = d["params"][nm]
        else:
            a, b = d["kernels"][nm]
            if a == b:
                ns[nm] = 1.0
            else:
                sigma = 0.5 * (bands[a] + bands[b])
                ns[nm] = xp.exp(-((bands[a] - bands[b]) ** 2) / (2.0 * sigma ** 2))
    # The formula was reduced to plain arithmetic on these names by _compile_formula(), so nothing else can
    # be reached from it. (An emptied __builtins__ here would break CuPy, which imports while it computes.)
    return eval(d["code"], {}, ns)


# ---------- compute engine: CuPy on a GPU, NumPy otherwise ----------
def pick_engine():
    if USE_GPU in ("auto", True):
        try:
            import cupy as cp
            if cp.cuda.runtime.getDeviceCount() > 0:
                _ = float(cp.arange(4, dtype=cp.float32).sum())      # forces the CUDA context
                name = cp.cuda.runtime.getDeviceProperties(0)["name"]
                return cp, f"GPU ({name.decode() if isinstance(name, bytes) else name}, CuPy {cp.__version__})"
        except Exception as e:
            if USE_GPU is True:
                raise PipelineStop(f"USE_GPU is True but no GPU could be used: {e}")
    return np, f"CPU (NumPy {np.__version__})"


XP, ENGINE = pick_engine()
CDTYPE = np.float32                                  # reflectance and index values are computed in 32-bit floats
if any(name not in INDICES for name in SPREAD_INDICES):
    raise ValueError("SPREAD_INDICES must be taken from INDICES")
log.info("Engine: %s | %d indices: %s | harmonisation of Landsat 5/7 to 8/9: %s | aerosol screen: %s | Landsat 7 "
         "until: %s", ENGINE, len(INDICES), " ".join(INDICES), "ON (Roy et al. 2016)" if HARMONIZE else "off",
         "on" if AEROSOL_SCREEN else "off", LANDSAT7_LAST_DAY or "the end")

In [ ]:
# ═══════════════ CELL 4: disk floor, verified Drive copy, logs ═══════════════
_LOG_LOCK   = threading.Lock()
_DRIVE_LOCK = threading.Lock()
DRIVE_SETTLE_S = 10                      # let the Drive FUSE mount settle before re-reading
TIMES = {}                               # seconds spent per stage (reported at the end of the run)
TOTALS = {}                              # counters (bytes downloaded, files, records) for the summary


class timed:
    """with timed("download"): ...  adds the elapsed seconds to TIMES["download"]."""

    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *exc):
        TIMES[self.name] = TIMES.get(self.name, 0.0) + time.time() - self.t0
        return False


def log_events(name, entries):
    """Append JSON lines to a log on Drive in one write + fsync (survives a dead runtime)."""
    if not entries:
        return
    stamp = iso(utcnow())
    text = "".join(json.dumps(dict(e, logged=stamp), default=str) + "\n" for e in entries)
    with _LOG_LOCK:
        with open(os.path.join(DRIVE_OUT, name), "a") as f:
            f.write(text)
            f.flush()
            os.fsync(f.fileno())


def human(n):
    n = float(n)
    for u in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024:
            return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"


# ---------- 20 GB scratch floor ----------
def free_gb():
    return shutil.disk_usage(DISK_PATH).free / 1024 ** 3


def ensure_disk_free(context="", extra_gb=0.0):
    """Block until MIN_FREE_GB + extra_gb is free. Scratch only ever holds the file being built,
    so a shortfall normally means Drive's local upload cache is still draining: wait for it
    rather than fail. Raises PipelineStop if it never clears."""
    need = MIN_FREE_GB + max(0.0, extra_gb)
    t0, warned = time.time(), False
    while True:
        free = free_gb()
        if free >= need:
            if warned:
                log.info("Disk OK again: %.1f GB free (need %.1f)", free, need)
            return free
        if not warned:
            warned = True
            log.warning("LOW DISK %.1f < %.1f GB (%s) - waiting for Drive to upload", free, need, context)
        if time.time() - t0 > DISK_MAX_WAIT_S:
            raise PipelineStop(f"Free space stayed at {free:.1f} GB (< {need:.1f} GB) for "
                               f"{DISK_MAX_WAIT_S // 60} min ({context}). Nothing is lost - re-run to resume.")
        time.sleep(60)


# ---------- verified copies to / from Drive ----------
def md5sum(path, block=10 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(block)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def robust_drive_copy(src, dst, max_retries=3):
    """Copy src to Drive under a temporary name, re-read it, compare size + MD5, then
    rename over dst. A runtime that dies mid-copy leaves the previous good dst intact."""
    if os.path.abspath(src) == os.path.abspath(dst):
        raise ValueError(f"robust_drive_copy: source and destination are the same file ({src})")
    with _DRIVE_LOCK, timed("Drive copy"):
        size = os.path.getsize(src)
        ensure_disk_free("Drive copy", extra_gb=size / 1024 ** 3)   # FUSE caches the upload locally
        want = md5sum(src)
        tmp = dst + ".tmp"
        os.makedirs(os.path.dirname(dst) or ".", exist_ok=True)
        last = None
        for attempt in range(1, max_retries + 1):
            try:
                with open(src, "rb") as fs, open(tmp, "wb") as fd:
                    shutil.copyfileobj(fs, fd, length=10 * 1024 * 1024)
                    fd.flush()
                    os.fsync(fd.fileno())
                ok = False
                for _ in range(4):           # a just-closed FUSE file can return stale bytes
                    time.sleep(DRIVE_SETTLE_S)
                    try:
                        if os.path.getsize(tmp) == size and md5sum(tmp) == want:
                            ok = True
                            break
                    except OSError as e:
                        last = e
                if not ok:
                    raise IOError("size or MD5 mismatch after settle")
                os.replace(tmp, dst)
                log.info("Drive copy verified (%s): %s", human(size), os.path.basename(dst))
                return True
            except Exception as e:
                last = e
                log.warning("Drive copy attempt %d/%d failed: %s", attempt, max_retries, e)
                try:
                    if os.path.exists(tmp):
                        os.remove(tmp)
                except OSError:
                    pass
                if attempt < max_retries:
                    time.sleep(10 * attempt)
        raise RuntimeError(f"Drive copy of {os.path.basename(src)} failed after {max_retries} attempts: {last}")


def restore_from_drive(src, dst):
    """Bring a Drive file back to scratch (size-checked, atomic)."""
    size = os.path.getsize(src)
    ensure_disk_free("restore from Drive", extra_gb=size / 1024 ** 3)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    tmp = dst + ".part"
    shutil.copyfile(src, tmp)
    if os.path.getsize(tmp) != size:
        os.remove(tmp)
        raise IOError(f"restore of {os.path.basename(src)} is incomplete")
    os.replace(tmp, dst)
    log.info("Restored %s from Drive (%s)", os.path.basename(src), human(size))

In [ ]:
# ═══════════════ CELL 5: aquifer polygon, scene list, the tile grid of each UTM zone, block plan ═══════════════
_ROI_GDF = gpd.read_file(SHP_PATH)
if _ROI_GDF.crs is None:
    raise ValueError(f"{SHP_PATH} has no CRS")
_POLY = {}


def roi_polygon(crs):
    """The aquifer polygon in a given CRS (one copy per CRS). The shapefile's parts are merged once,
    in its own CRS; the merged outline is then reprojected, which is quick."""
    if None not in _POLY:
        g = _ROI_GDF.geometry
        u = g.union_all() if hasattr(g, "union_all") else g.unary_union
        _POLY[None] = u if u.is_valid else u.buffer(0)
    if crs not in _POLY:
        u = gpd.GeoSeries([_POLY[None]], crs=_ROI_GDF.crs).to_crs(crs).iloc[0]
        _POLY[crs] = u if u.is_valid else u.buffer(0)
    return _POLY[crs]


def _rough_outline():
    """A rough lon/lat outline, used only as the spatial filter of the catalogue listing."""
    rough = roi_polygon("EPSG:4326").buffer(0.15).simplify(0.10, preserve_topology=True)
    if rough.geom_type != "Polygon":
        rough = rough.convex_hull
    return json.dumps(shapely.geometry.mapping(shapely.geometry.Polygon(rough.exterior)))


ROI_REGION = _rough_outline()
log.info("Aquifer polygon: %.0f km2", roi_polygon("EPSG:5070").area / 1e6)


def with_retries(fn, what, tries=None, stop=None):
    """Run an Earth Engine request; retry with growing pauses. A missing asset is not retried.
    `stop` (a threading.Event) ends the waiting at once when the run is being stopped."""
    n = 1 + (MAX_RETRIES if tries is None else tries)
    last = None
    for attempt in range(n):
        if stop is not None and stop.is_set():
            raise RunStopping(what)
        try:
            return fn()
        except Exception as e:
            last = e
            msg = str(e)
            low = msg.lower()
            if "request size" in low and "must be less than" in low:
                raise RequestTooLarge(msg[:200])
            if "asset" in low and ("not found" in low or "does not exist" in low):
                raise SceneUnavailable(f"{what}: {msg[:200]}")
            if attempt + 1 < n:
                quota = "quota" in low or "429" in msg or "too many" in low
                pause = min((20 if quota else 2) * 2 ** attempt, 300)
                if stop is None:
                    time.sleep(pause)
                elif stop.wait(pause):
                    raise RunStopping(what)
    raise IOError(f"{what}: {type(last).__name__}: {str(last)[:300]}")


# ---------- which months ----------
def parse_month(text, what):
    """ "YYYY-MM" -> (year, month)."""
    m = re.fullmatch(r"(\d{4})-(\d{1,2})", str(text).strip())
    if not m or not 1 <= int(m.group(2)) <= 12:
        raise ValueError(f'{what} = {text!r} is not a month. Write it as "YYYY-MM", for example "2005-01".')
    return int(m.group(1)), int(m.group(2))


FIRST_MONTH = parse_month(START_MONTH, "START_MONTH")
LAST_MONTH  = parse_month(STOP_MONTH, "STOP_MONTH") if STOP_MONTH is not None else None
if LAST_MONTH is not None and LAST_MONTH < FIRST_MONTH:
    raise ValueError(f"STOP_MONTH ({STOP_MONTH}) is before START_MONTH ({START_MONTH}).")


def month_bounds(year, month):
    start = datetime(year, month, 1)
    return start, datetime(year + (month == 12), month % 12 + 1, 1)


def ym(year, month):
    return f"{year}-{month:02d}"


def span(months):
    """[1, 2, 3, 7, 8] -> '1-3, 7-8'."""
    out, first = [], None
    for i, m in enumerate(months):
        first = m if first is None else first
        if i + 1 == len(months) or months[i + 1] != m + 1:
            out.append(str(m) if first == m else f"{first}-{m}")
            first = None
    return ", ".join(out)


def range_months():
    """Every (year, month) from START_MONTH to STOP_MONTH that is ready, in order. A month is ready
    SETTLE_DAYS after it ended; the range ends at the first month that is not."""
    out, (y, m), now = [], FIRST_MONTH, utcnow()
    while (LAST_MONTH is None or (y, m) <= LAST_MONTH) and month_bounds(y, m)[1] + timedelta(days=SETTLE_DAYS) <= now:
        out.append((y, m))
        y, m = y + (m == 12), m % 12 + 1
    return out


_JOB = {}


def job_months(refresh=False):
    """The (year, month) list of this run: in the rehearsal the chosen months, otherwise
    range_months(). Worked out once per run, so the list cannot change while files are written."""
    if refresh or "months" not in _JOB:
        months = [(TEST_YEAR, m) for m in sorted(set(TEST_MONTHS))] if TEST_MODE else range_months()
        by_year = {}
        for y, m in months:
            by_year.setdefault(y, []).append(m)
        _JOB.update(months=months, by_year=by_year)
    return _JOB["months"]


def year_list():
    job_months()
    return sorted(_JOB["by_year"])


def wanted_months(year):
    """Months of a year that this run is to have on Drive."""
    job_months()
    return _JOB["by_year"].get(year, [])


# ---------- scene list (cached on Drive) ----------
def _scene_record(sat, im):
    """One catalogue entry -> what the pipeline needs. Returns (record, None) or (None, reason)."""
    p = im.get("properties", {})
    bands = {b["id"]: b for b in im.get("bands", [])}
    need = QA_BANDS + sorted(set(BAND_LETTERS[SENSOR[sat]].values())) + \
        ([AEROSOL_BAND[SENSOR[sat]][0]] if AEROSOL_SCREEN else [])
    if any(k not in bands for k in need):
        return None, f"bands missing: {[k for k in need if k not in bands]}"
    g = bands["QA_PIXEL"]["grid"]
    if any(bands[k]["grid"] != g for k in need):
        return None, "bands are not on one common grid"
    at = g["affineTransform"]
    if at.get("shearX", 0) or at.get("shearY", 0) or "WRS_PATH" not in p or "WRS_ROW" not in p:
        return None, "rotated grid or no WRS path/row"
    return dict(id=im["id"].split("/")[-1], name=im["name"], sat=sat, t=im["startTime"],
                path=int(p["WRS_PATH"]), row=int(p["WRS_ROW"]), cloud=float(p.get("CLOUD_COVER", -1)),
                product=str(p.get("LANDSAT_PRODUCT_ID", "")), crs=g["crsCode"], tx=float(at["translateX"]),
                ty=float(at["translateY"]), sx=float(at["scaleX"]), sy=float(at["scaleY"]),
                w=int(g["dimensions"]["width"]), h=int(g["dimensions"]["height"])), None


def _list_collection(sat, t0, t1):
    parent = f"projects/earthengine-public/assets/{collection_id(sat)}"
    recs, odd, token = [], [], None
    while True:
        params = dict(parent=parent, region=ROI_REGION, pageSize=1000, view="FULL",
                      startTime=f"{t0}T00:00:00Z", endTime=f"{t1}T00:00:00Z")
        if token:
            params["pageToken"] = token
        page = with_retries(lambda: ee.data.listImages(params), f"listing {sat} {t0[:4]}")
        for im in page.get("images", []):
            rec, why = _scene_record(sat, im)
            if rec is None:
                odd.append(dict(scene=im.get("id", "?"), reason=why))
            else:
                recs.append(rec)
        token = page.get("nextPageToken")
        if not token:
            return recs, odd


def get_manifest():
    """Every scene of the chosen satellites and years that touches the aquifer's outline, with its grid."""
    years = year_list()
    sig = json.dumps(dict(sats=SATELLITES, tier=TIER, years=[years[0], years[-1]]), sort_keys=True)
    path = os.path.join(DRIVE_OUT, MANIFEST_NAME)
    if os.path.exists(path) and not REBUILD_MANIFEST:
        try:
            with gzip.open(path, "rt") as f:
                m = json.load(f)
            age = (utcnow() - parse_iso(m["created"])).total_seconds() / 86400
            if m.get("signature") == sig and age <= MANIFEST_MAX_AGE_DAYS:
                log.info("Scene list from Drive: %d scenes, %.1f days old", len(m["scenes"]), age)
                return m["scenes"]
        except Exception as e:
            log.warning("Cached scene list unreadable (%s) - listing again", e)
    t0 = time.time()
    jobs = [(s, f"{y}-01-01", f"{y + 1}-01-01") for s in SATELLITES for y in years]
    recs, odd = [], []
    with ThreadPoolExecutor(min(8, len(jobs))) as ex:
        for r, o in ex.map(lambda j: _list_collection(*j), jobs):
            recs += r
            odd += o
    seen, uniq = set(), []
    for r in sorted(recs, key=lambda r: (r["t"], r["id"])):
        if r["id"] not in seen:
            seen.add(r["id"])
            uniq.append(r)
    log_events(LOG_SKIPPED, [dict(o, stage="scene list") for o in odd])
    local = os.path.join(NC_DIR, MANIFEST_NAME)
    with gzip.open(local, "wt") as f:
        json.dump(dict(created=iso(utcnow()), signature=sig, scenes=uniq), f)
    robust_drive_copy(local, path)
    os.remove(local)
    log.info("Scene list: %d scenes (%s), %d unusable entries, %.0f s", len(uniq),
             ", ".join(f"{s} {sum(1 for r in uniq if r['sat'] == s)}" for s in SATELLITES), len(odd), time.time() - t0)
    return uniq


# ---------- the tile grid of each UTM zone ----------
# Every Landsat scene of a UTM zone lies on one 30 m lattice, whatever its path/row; scenes differ only
# by shifts of whole pixels. Each zone is therefore cut into square tiles of that lattice, and a tile
# takes the pixels of every scene of its zone that covers it. Where two paths overlap, a pixel so gets
# one monthly value from all the passes that saw it, and nothing is resampled. A tile's frame is the
# tile cut to the aquifer; it depends on geometry only, is stored on Drive and never changes, so every
# yearly file of a tile has the same x and y.
TILE_M = TILE_PX * PIXEL


def tile_key(crs, i, j):
    return f"z{int(crs.split(':')[1]) % 100:02d}e{i:02d}n{j:02d}"


def _mask_path(key):
    return os.path.join(MASK_DIR, f"{key}.npy")


def _rasterize(F):
    return rasterio.features.geometry_mask([roi_polygon(F["crs"])], out_shape=(F["h"], F["w"]), invert=True,
                                           all_touched=False,          # pixel centre inside the polygon
                                           transform=Affine(PIXEL, 0.0, F["x0"], 0.0, -PIXEL, F["y0"]))


def _save_mask(key, mask):
    tmp = _mask_path(key) + ".part"
    with open(tmp, "wb") as f:
        np.save(f, np.packbits(mask))
    os.replace(tmp, _mask_path(key))


def frame_mask(key, F):
    """True where the pixel centre lies inside the aquifer polygon (kept on scratch, 1 bit per pixel)."""
    if os.path.exists(_mask_path(key)):
        bits = np.load(_mask_path(key))
        if bits.size * 8 >= F["h"] * F["w"]:
            mask = np.unpackbits(bits)[:F["h"] * F["w"]].reshape(F["h"], F["w"]).astype(bool)
            if int(mask.sum()) == F["n_roi"]:
                return mask
    mask = _rasterize(F)
    if int(mask.sum()) != F["n_roi"]:
        raise DataIntegrityError(f"{key}: the aquifer mask has {int(mask.sum())} pixels, its frame says {F['n_roi']}. "
                                 "The shapefile differs from the one the frames were made with.")
    _save_mask(key, mask)
    return mask


def on_lattice(r, ref):
    return (r["crs"] == ref["crs"] and r["sx"] == PIXEL and r["sy"] == -PIXEL
            and (r["tx"] - ref["tx"]) % PIXEL == 0 and (r["ty"] - ref["ty"]) % PIXEL == 0)


def make_frame(key, crs, ox, oy, i, j):
    """Frame of tile (i, j) of a zone whose lattice has its pixel edges at ox, oy (mod 30 m): the
    tile cut to the rectangle that bounds its aquifer pixels. None when it holds no aquifer pixel."""
    left, top = ox + i * TILE_M, oy + (j + 1) * TILE_M
    box = shapely.geometry.box(left, top - TILE_M, left + TILE_M, top)
    if not roi_polygon(crs).intersects(box):
        return None
    F = dict(crs=crs, x0=left, y0=top, w=TILE_PX, h=TILE_PX)
    mask = _rasterize(F)
    if not mask.any():
        return None
    rows, cols = np.flatnonzero(mask.any(axis=1)), np.flatnonzero(mask.any(axis=0))
    r0, r1, c0, c1 = int(rows[0]), int(rows[-1]) + 1, int(cols[0]), int(cols[-1]) + 1
    if TEST_MODE and TEST_FRAME_PX:                       # a small square around the aquifer pixel nearest the centre
        rr, cc = np.nonzero(mask[r0:r1:8, c0:c1:8])
        if not rr.size:
            return None
        k = int(np.argmin((rr * 8 - (r1 - r0) / 2) ** 2 + (cc * 8 - (c1 - c0) / 2) ** 2))
        ra = max(r0, r0 + int(rr[k]) * 8 - TEST_FRAME_PX // 2)
        ca = max(c0, c0 + int(cc[k]) * 8 - TEST_FRAME_PX // 2)
        r0, r1, c0, c1 = ra, min(r1, ra + TEST_FRAME_PX), ca, min(c1, ca + TEST_FRAME_PX)
    sub = np.ascontiguousarray(mask[r0:r1, c0:c1])
    F = dict(crs=crs, x0=left + c0 * PIXEL, y0=top - r0 * PIXEL, w=c1 - c0, h=r1 - r0, n_roi=int(sub.sum()))
    _save_mask(key, sub)
    return F


def get_frames(cand):
    """Frames of the candidate tiles {key: (crs, ox, oy, i, j)}: read from Drive, and made (then
    stored) for tiles that have none yet."""
    path = os.path.join(DRIVE_OUT, FRAMES_NAME)
    tag = dict(pixel=PIXEL, tile_px=TILE_PX, test_frame_px=TEST_FRAME_PX if TEST_MODE else None)
    stored = {}
    if os.path.exists(path):
        with open(path) as f:
            doc = json.load(f)
        if doc.get("made_with") != tag:
            if not TEST_MODE:
                raise PipelineStop(f"{FRAMES_NAME} on Drive was made with other settings ({doc.get('made_with')}); "
                                   "this run would not fit the files next to it. Use an empty output folder.")
            doc = {}
        stored = doc.get("frames", {})
    new, t0 = 0, time.time()
    for key in sorted(cand):
        if key not in stored:
            stored[key] = make_frame(key, *cand[key])       # None is stored too: "no aquifer pixel here"
            new += 1
    if new:
        local = os.path.join(NC_DIR, FRAMES_NAME)
        with open(local, "w") as f:
            json.dump(dict(made_with=tag, created=iso(utcnow()), frames=stored), f, indent=1)
        robust_drive_copy(local, path)
        os.remove(local)
        log.info("Frames made for %d tiles in %.0f s and stored on Drive", new, time.time() - t0)
    return {k: stored[k] for k in cand if stored.get(k)}


def build_tiles(recs):
    """Tiles with aquifer pixels: their frame and, by (year, month), every scene of their UTM zone
    that covers them, whatever its path/row."""
    by_zone = {}
    for r in recs:
        by_zone.setdefault(r["crs"], []).append(r)
    cand, touching, odd = {}, {}, []
    for crs, scenes in sorted(by_zone.items()):
        ref = scenes[0]
        ox, oy = ref["tx"] % PIXEL, ref["ty"] % PIXEL
        for r in scenes:
            if r["sat"] == "LE07" and LANDSAT7_LAST_DAY and r["t"][:10] > LANDSAT7_LAST_DAY:
                continue                         # left out by the LANDSAT7_LAST_DAY setting
            if not on_lattice(r, ref):
                odd.append(dict(scene=r["id"], stage="not on its zone's 30 m grid"))
                continue
            i0, i1 = math.floor((r["tx"] - ox) / TILE_M), math.ceil((r["tx"] + r["w"] * PIXEL - ox) / TILE_M) - 1
            j0, j1 = math.floor((r["ty"] - r["h"] * PIXEL - oy) / TILE_M), math.ceil((r["ty"] - oy) / TILE_M) - 1
            for i in range(i0, i1 + 1):
                for j in range(j0, j1 + 1):
                    key = tile_key(crs, i, j)
                    cand[key] = (crs, ox, oy, i, j)
                    touching.setdefault(key, []).append(r)
    log_events(LOG_SKIPPED, odd)
    out = []
    for key, F in get_frames(cand).items():
        months = {}
        for r in sorted(touching[key], key=lambda r: (r["t"], r["id"])):
            if scene_window(r, F):
                months.setdefault((int(r["t"][:4]), int(r["t"][5:7])), []).append(r)
        if months:
            out.append(dict(key=key, F=F, months=months))
    out.sort(key=lambda P: (-P["F"]["n_roi"], P["key"]))
    return out


# ---------- where a scene lies on the frame, and its download blocks ----------
def scene_window(r, F):
    """The part of scene r that falls on frame F: dict(fr0, fc0 = its corner on the frame, h, w,
    sr0, sc0 = the same corner in the scene's own pixel grid). {} when they do not overlap, None when
    the scene is not on the frame's lattice (it is then left out, never resampled)."""
    dc, dr = (r["tx"] - F["x0"]) / PIXEL, (F["y0"] - r["ty"]) / PIXEL
    if r["crs"] != F["crs"] or r["sx"] != PIXEL or r["sy"] != -PIXEL or dc != int(dc) or dr != int(dr):
        return None
    dc, dr = int(dc), int(dr)
    fr0, fr1, fc0, fc1 = max(0, dr), min(F["h"], dr + r["h"]), max(0, dc), min(F["w"], dc + r["w"])
    if fr0 >= fr1 or fc0 >= fc1:
        return {}
    return dict(fr0=fr0, fc0=fc0, h=fr1 - fr0, w=fc1 - fc0, sr0=fr0 - dr, sc0=fc0 - dc)


# Earth Engine refuses a pixel request whose size, counted as pixels x bands x 3 bytes, is above 48 MiB.
# Both numbers were read from its own error message and checked against the service: 1448 x 1448 pixels
# of 8 bands (50,320,896 bytes) are served, 1449 x 1448 (50,355,648 bytes) are refused.
EE_REQUEST_BYTES   = 48 * 1024 ** 2
EE_BYTES_PER_VALUE = 3
N_REFL = 6
N_BANDS = len(QA_BANDS) + N_REFL + bool(AEROSOL_SCREEN)   # two quality bands, six reflective bands, the aerosol band


def max_block_pixels():
    """Most pixels one download block may hold: the BLOCK_PX setting, cut back to Earth Engine's limit."""
    return max(1, min(int(BLOCK_PX) ** 2, EE_REQUEST_BYTES // (N_BANDS * EE_BYTES_PER_VALUE)))


def tile_scene(h, w, max_px):
    """Cut an h x w pixel rectangle into few equal rectangles of at most max_px pixels each; returns
    (row edges, column edges). Rectangles rather than squares: a 7991 x 7251 scene takes 28
    requests this way and 36 as squares, and the time goes into the number of requests. Long thin
    strips are avoided (side ratio at most 2) when that costs no more than a couple of requests,
    because compact blocks are more often skipped whole where the scene leaves the aquifer."""
    cands = []
    for nr in range(1, h + 1):
        bh = -(-h // nr)                         # tallest block with nr rows
        if bh <= max_px:
            nc = -(-w // (max_px // bh))         # fewest columns that keep bh x width within max_px
            bw = -(-w // nc)
            cands.append((nr * nc, max(bh, bw) / min(bh, bw), nr, nc))
            if nc == 1 or nr > 2 * min(c[0] for c in cands):
                break
    fewest = min(c[0] for c in cands)
    compact = [c for c in cands if c[1] <= 2.0 and c[0] <= fewest + max(2, fewest // 8)]
    _, _, nr, nc = min(compact or [c for c in cands if c[0] == fewest])
    return [i * h // nr for i in range(nr + 1)], [j * w // nc for j in range(nc + 1)]


def plan_blocks(S, mask):
    """Download blocks of a scene window: rectangles (row0, col0, h, w), in the window's own pixel
    count, that hold at least one aquifer pixel."""
    rows, cols = tile_scene(S["h"], S["w"], max_block_pixels())
    return [(a, c, b - a, d - c) for a, b in zip(rows, rows[1:]) for c, d in zip(cols, cols[1:])
            if mask[S["fr0"] + a:S["fr0"] + b, S["fc0"] + c:S["fc0"] + d].any()]

In [ ]:
# ═══════════════ CELL 6: download the 30 m pixels of one month to scratch, check every block twice ═══════════════
def _grid(r, row0, col0, h, w):
    """A pixel rectangle of scene r on the scene's own grid: nothing is resampled on the way."""
    return dict(dimensions=dict(width=int(w), height=int(h)),
                affineTransform=dict(scaleX=r["sx"], shearX=0, translateX=r["tx"] + col0 * r["sx"],
                                     shearY=0, scaleY=r["sy"], translateY=r["ty"] + row0 * r["sy"]),
                crsCode=r["crs"])


def _check_block(arr, bands, shape):
    """A downloaded block must be exactly what was asked for: these bands, this shape, unsigned
    16-bit (the aerosol band: its own stored type)."""
    kinds = dict(AEROSOL_BAND.values())
    if arr.dtype.names != tuple(bands):
        raise ValueError(f"bands {arr.dtype.names} instead of {tuple(bands)}")
    if any(arr.dtype[b] != np.dtype(kinds.get(b, "<u2")) for b in bands):
        raise ValueError(f"unexpected data type {arr.dtype}")
    if arr.shape != tuple(shape):
        raise ValueError(f"shape {arr.shape} instead of {tuple(shape)}")
    return arr


def scene_bands(r):
    """Band order of a scene's scratch file: the two quality bands, the six reflective bands, and
    (with AEROSOL_SCREEN) the aerosol band."""
    return QA_BANDS + sorted(set(BAND_LETTERS[SENSOR[r["sat"]]].values())) + \
        ([AEROSOL_BAND[SENSOR[r["sat"]]][0]] if AEROSOL_SCREEN else [])


_SPLIT_NOTED = [False]


def fetch_array(r, bands, row0, col0, h, w, stop=None):
    """Earth Engine -> memory: one rectangle of scene r as the stored integers (getPixels, so no
    Earth Engine computation is involved). Returns (array, bytes received). A rectangle that Earth
    Engine refuses for its size is fetched as two halves and put together again, so a wrong guess
    about its limit costs a few extra requests and never the run."""
    def once():
        raw = ee.data.getPixels(dict(assetId=r["name"], bandIds=list(bands), grid=_grid(r, row0, col0, h, w),
                                     fileFormat="NPY"))
        return _check_block(np.load(io.BytesIO(raw), allow_pickle=False), bands, (h, w)), len(raw)   # refuse a damaged reply

    try:
        return with_retries(once, f"{r['id']} block {row0},{col0}", stop=stop)
    except RequestTooLarge as e:
        if h * w < 4096:
            raise IOError(f"{r['id']}: Earth Engine refuses even a {h} x {w} pixel request ({e})")
        if not _SPLIT_NOTED[0]:
            _SPLIT_NOTED[0] = True
            log.warning("Earth Engine refused a %d x %d pixel request of %d bands for its size (%s). Such blocks "
                        "are fetched in halves from now on; lowering BLOCK_PX avoids the extra requests.",
                        h, w, len(bands), e)
        if h >= w:
            k = h // 2
            top, n1 = fetch_array(r, bands, row0, col0, k, w, stop)
            bottom, n2 = fetch_array(r, bands, row0 + k, col0, h - k, w, stop)
            return np.concatenate([top, bottom], axis=0), n1 + n2
        k = w // 2
        left, n1 = fetch_array(r, bands, row0, col0, h, k, stop)
        right, n2 = fetch_array(r, bands, row0, col0 + k, h, w - k, stop)
        return np.concatenate([left, right], axis=1), n1 + n2


def download_block(r, S, blk, mm, stop):
    """One block of one scene: fetch its bands in one request and put them in the scene's scratch
    file, which holds 16 bits per value (the 8-bit aerosol band is widened, the signed opacity band
    keeps its bits). Returns the block with a checksum of what was written."""
    row0, col0, h, w = blk
    bands = scene_bands(r)
    arr, nbytes = fetch_array(r, bands, S["sr0"] + row0, S["sc0"] + col0, h, w, stop)
    if stop.is_set():                        # the scratch folder may already belong to the next run
        raise RunStopping(r["id"])
    crc = 0
    for k, b in enumerate(bands):
        plane = np.ascontiguousarray(arr[b])
        plane = plane.view("<u2") if plane.dtype == np.dtype("<i2") else plane.astype("<u2", copy=False)
        crc = zlib.crc32(plane, crc)
        mm[k, row0:row0 + h, col0:col0 + w] = plane
    return dict(blk=blk, crc=crc, bytes=nbytes)


def verify_scene(path, S, blocks):
    """Read a scene's scratch file back and compare every block with the checksum taken on arrival."""
    mm = np.load(path, mmap_mode="r")
    try:
        if mm.shape != (N_BANDS, S["h"], S["w"]) or mm.dtype != np.dtype("<u2"):
            raise IOError(f"{os.path.basename(path)}: unexpected layout {mm.dtype} {mm.shape}")
        for b in blocks:
            row0, col0, h, w = b["blk"]
            crc = 0
            for k in range(N_BANDS):
                crc = zlib.crc32(np.ascontiguousarray(mm[k, row0:row0 + h, col0:col0 + w]), crc)
            if crc != b["crc"]:
                raise IOError(f"{os.path.basename(path)}: block {row0},{col0} read back differently from what arrived")
    finally:
        del mm


def download_scenes(tag, scenes, F, mask, stop):
    """All blocks of the given scenes, N_DOWNLOAD_THREADS at a time, each scene into one scratch
    file on its part of the frame. Returns (ok, permanent, transient): ok = [(scene, part)], the
    others = [(scene, reason)]. A scene counts only when every block arrived, passed its check and
    was read back from scratch unchanged. Pixels that are not fetched (blocks without an aquifer
    pixel) are preset to 'fill', so they can never be taken for data."""
    plans = {}
    for r in scenes:
        S = scene_window(r, F)
        plans[r["id"]] = (S, plan_blocks(S, mask))
    ensure_disk_free(f"{tag} download", extra_gb=sum(S["h"] * S["w"] for S, _ in plans.values()) * 2 * N_BANDS / 1024 ** 3)
    folder = os.path.join(RAW_DIR, tag)
    os.makedirs(folder, exist_ok=True)
    got, errors, futs, mms, paths = {r["id"]: [] for r in scenes}, {}, {}, {}, {}
    ex = ThreadPoolExecutor(N_DOWNLOAD_THREADS)
    try:
        with timed("download"):
            for r in scenes:
                S, blocks = plans[r["id"]]
                if not blocks:
                    continue
                paths[r["id"]] = os.path.join(folder, r["id"] + ".npy")
                mm = np.lib.format.open_memmap(paths[r["id"]], mode="w+", dtype="<u2", shape=(N_BANDS, S["h"], S["w"]))
                mm[0] = 1                        # QA_PIXEL = fill everywhere until a block says otherwise
                mms[r["id"]] = mm
                for blk in blocks:
                    futs[ex.submit(download_block, r, S, blk, mm, stop)] = r
            for fut in as_completed(futs):
                if stop.is_set():
                    raise RunStopping(tag)
                r = futs[fut]
                try:
                    got[r["id"]].append(fut.result())
                except SceneUnavailable as e:        # only Earth Engine saying "no such scene" is permanent
                    errors[r["id"]] = ("permanent", str(e))
                except Exception as e:
                    errors.setdefault(r["id"], ("transient", f"{type(e).__name__}: {e}"))
    except BaseException:
        stop.set()
        ex.shutdown(wait=False, cancel_futures=True)
        raise
    ex.shutdown(wait=True)
    for mm in mms.values():
        mm.flush()
    mm = fut = None                              # nothing may keep a scratch file open from here on: a failed
    mms.clear()                                  # request's traceback would, through the future that holds it
    futs.clear()
    gc.collect()
    ok, permanent, transient = [], [], []
    for r in scenes:
        S, blocks = plans[r["id"]]
        path = paths.get(r["id"])
        if r["id"] not in errors and blocks:
            try:
                with timed("download"):
                    verify_scene(path, S, got[r["id"]])
            except Exception as e:
                errors[r["id"]] = ("transient", f"{type(e).__name__}: {e}")
        if r["id"] in errors:
            kind, why = errors[r["id"]]
            (permanent if kind == "permanent" else transient).append((r, why))
            if path and os.path.exists(path):
                os.remove(path)
        elif blocks:
            nbytes = sum(b["bytes"] for b in got[r["id"]])
            TOTALS["bytes"] = TOTALS.get("bytes", 0) + nbytes
            TOTALS["blocks"] = TOTALS.get("blocks", 0) + len(blocks)
            ok.append((r, dict(S, path=path, bands=scene_bands(r), bytes=nbytes, blocks=len(blocks))))
        else:
            ok.append((r, None))                 # the scene touches the frame but none of its aquifer pixels
    return ok, permanent, transient


def fetch_month(tag, scenes, F, mask, stop):
    """Every scene of one month that covers one tile, on scratch. A scene that cannot be completed is
    tried again with growing pauses; after STALL_MINUTES without any progress the run stops
    cleanly. Returns (parts, permanent): parts = [(scene, part)] in time order."""
    todo = [r for r in scenes if r["id"] not in SKIP_SCENES]
    permanent = [(r, "listed in SKIP_SCENES") for r in scenes if r["id"] in SKIP_SCENES]
    parts, waits, since = [], 0, None
    while todo:
        ok, perm, transient = download_scenes(tag, todo, F, mask, stop)
        parts += ok
        permanent += perm
        todo = [r for r, _ in transient]
        if ok or perm:
            waits, since = 0, None
        if todo:
            r0_, why0 = min(transient, key=lambda x: x[0]["t"])
            log.warning("[%s] %d scene(s) could not be completed; first: %s (%s)", tag, len(todo), r0_["id"], why0[:200])
            since = since or time.time()
            if time.time() - since > STALL_MINUTES * 60:
                raise PipelineStop(
                    f"No progress for {STALL_MINUTES} min; stuck at {r0_['id']} ({why0[:200]}). Everything on Drive "
                    "is safe - re-run to resume. If this same scene blocks every run while Earth Engine is "
                    "otherwise fine, add its id to SKIP_SCENES.")
            waits += 1
            if stop.wait(min(30 * 2 ** waits, 600)):
                raise RunStopping(tag)
    return sorted(parts, key=lambda x: (x[0]["t"], x[0]["id"])), permanent

In [ ]:
# ═══════════════ CELL 7: reflectance, indices, monthly median, 16-bit packing; engine and eemont checks ═══════════════
SCALE = {}        # satellite -> {band: (scale, offset)}, filled from eemont in load_scale_params()
LETTERS_USED = sorted({L for d in INDEX_DEFS.values() for L in d["letters"]})
FILL = -32768     # _FillValue of the index variables
NO_OBS = 255      # _FillValue of n_obs (pixels outside the aquifer)


def load_scale_params():
    """Scale and offset of every band, asked from eemont (its copy of the Earth Engine catalogue)."""
    for sat in SATELLITES:
        ic = ee.ImageCollection(collection_id(sat))
        sc, off = ic.getScaleParams(), ic.getOffsetParams()          # eemont
        bands = BAND_LETTERS[SENSOR[sat]].values()
        if sc is None or off is None or any(b not in sc or b not in off for b in bands):
            raise PipelineStop(f"eemont has no scale/offset for {collection_id(sat)}")
        SCALE[sat] = {b: (float(sc[b]), float(off[b])) for b in bands}
    log.info("Scale/offset from eemont: %s", {s: SCALE[s][BAND_LETTERS[SENSOR[s]]["N"]] for s in SATELLITES})


def _to_np(a):
    return a.get() if hasattr(a, "get") else np.asarray(a)


def qa_usable(xp, qa_pixel, qa_radsat):
    """Pixels that pass the quality rule: present, none of the rejected QA_PIXEL bits, not saturated."""
    ok = ((qa_pixel & 1) == 0) & ((qa_pixel & QA_PIXEL_REJECT) == 0)
    if REJECT_SATURATED:
        ok = ok & (qa_radsat == 0)
    return ok


def reflectance(xp, dn, sat, dtype=None):
    """Stored integers -> surface reflectance, keyed by band letter. With HARMONIZE, Landsat 4/5/7
    reflectance is then put on Landsat 8/9's scale band by band (Roy et al. 2016)."""
    dtype = CDTYPE if dtype is None else dtype
    letters, out = BAND_LETTERS[SENSOR[sat]], {}
    for L, v in dn.items():
        s, o = SCALE[sat][letters[L]]
        ref = v.astype(dtype) * dtype(s) + dtype(o)
        if HARMONIZE and SENSOR[sat] == "TM":
            slope, intercept = HARMONIZATION[L]
            ref = ref * dtype(slope) + dtype(intercept)
        out[L] = ref
    return out


def median_of(xp, stack, count):
    """Median along the first axis of `stack`, which holds +inf where an observation has no value;
    `count` is the number of real values per pixel. With an even count it is the mean of the two
    middle values. Pixels with no value give NaN. `stack` is used as work space and comes back sorted.

    A month has only a handful of observations, so the planes are sorted against each other with
    whole-array minimum / maximum steps (an odd-even transposition network). That is far quicker
    than a general sort along a short axis, on the GPU and on the CPU, and +inf ends up last on any
    engine, which NaN would not do reliably."""
    k = stack.shape[0]
    for rnd in range(k):
        for i in range(rnd % 2, k - 1, 2):
            lo = xp.minimum(stack[i], stack[i + 1])
            xp.maximum(stack[i], stack[i + 1], out=stack[i + 1])
            stack[i] = lo
    n = xp.maximum(count, 1)
    lo = xp.take_along_axis(stack, ((n - 1) // 2)[None], axis=0)[0]
    hi = xp.take_along_axis(stack, xp.minimum(n // 2, k - 1)[None], axis=0)[0]
    return xp.where(count > 0, (lo + hi) * xp.float32(0.5), xp.float32(np.nan))


def pack(xp, values, scale):
    """Index values -> 16-bit integers (index = integer x scale, rounded to the nearest step).
    No value, or a value outside what 16 bits can hold at this scale, becomes the fill value."""
    q = xp.rint(values.astype(xp.float64) / scale)
    ok = xp.isfinite(q) & (xp.abs(q) <= 32767)
    return xp.where(ok, q, FILL).astype(xp.int16)


def pass_of(r):
    """Scenes of one satellite pass: same satellite, same WRS-2 path, same day. Where two of them
    overlap (consecutive rows of the path) they show the same instant, so they count once."""
    return r["sat"], r["path"], r["t"][:10]


def hazy(xp, plane, sat):
    """Pixels the aerosol band calls hazy. Landsat 8/9: aerosol level "high" (bits 6-7 of
    SR_QA_AEROSOL = 3). Landsat 5/7: atmospheric opacity above OPACITY_MAX (stored x 1000 as signed
    integers; -9999 = not retrieved, which is not counted as hazy)."""
    if SENSOR[sat] == "OLI":
        return (plane >> 6) == 3
    return plane.astype(xp.int16) > int(round(OPACITY_MAX * 1000))


def compose_strip(parts, mask_strip, r0, r1, c0, c1, xp=None):
    """Monthly values of frame rows r0..r1, columns c0..c1.

    parts = [(scene, part)] of the month in time order; part["mm"] holds the scene's planes on its
    window of the frame. Scenes are laid on the frame by whole-pixel shifts. One observation of a
    pixel = one satellite pass in which the pixel is usable (the first scene of the pass that has
    it). For every index the value of a pixel is the median of its observations in the month.
    Returns (arrays, n_obs, counters, extra): arrays = {index: int16, <index>_range: int16} or None
    when no pass has a usable pixel here; n_obs as uint8; extra = {"n_obs_oli": how many of the
    observations are Landsat 8/9, "obs_day": their mean day of the month, rounded}, both uint8."""
    xp = XP if xp is None else xp
    h, w = r1 - r0, c1 - c0
    inside = xp.asarray(mask_strip)
    passes, taken, obs, info = {}, {}, [], {}
    px = dict(clear=0, out_of_range=0, unstorable=0, hazy=0)
    for r, part in parts:
        if part is None:
            continue
        a0, a1 = max(r0, part["fr0"]), min(r1, part["fr0"] + part["h"])      # overlap, in frame rows / columns
        b0, b1 = max(c0, part["fc0"]), min(c1, part["fc0"] + part["w"])
        if a0 >= a1 or b0 >= b1:
            continue
        dev = xp.asarray(np.ascontiguousarray(part["mm"][:, a0 - part["fr0"]:a1 - part["fr0"],
                                                         b0 - part["fc0"]:b1 - part["fc0"]]))
        win = (slice(a0 - r0, a1 - r0), slice(b0 - c0, b1 - c0))
        use = inside[win] & qa_usable(xp, dev[0], dev[1])
        if REQUIRE_VALID_RANGE:
            inr = use
            for k in range(len(QA_BANDS), len(QA_BANDS) + N_REFL):
                inr = inr & (dev[k] >= SR_VALID[0]) & (dev[k] <= SR_VALID[1])
            px["out_of_range"] += int((use & ~inr).sum())
            use = inr
        if AEROSOL_SCREEN:
            clear = use & ~hazy(xp, dev[N_BANDS - 1], r["sat"])
            px["hazy"] += int((use & ~clear).sum())
            use = clear
        p = passes.get(pass_of(r))
        if p is not None:
            use = use & ~taken[p][win]           # an earlier scene of this pass already supplied these pixels
        n_use = int(use.sum())
        if not n_use:
            continue
        if p is None:
            p = passes[pass_of(r)] = len(passes)
            taken[p] = xp.zeros((h, w), dtype=bool)
            info[p] = (SENSOR[r["sat"]] == "OLI", int(r["t"][8:10]))
        taken[p][win] |= use
        px["clear"] += n_use
        letters = BAND_LETTERS[SENSOR[r["sat"]]]
        dn = {L: dev[part["bands"].index(letters[L])] for L in LETTERS_USED}
        obs.append((p, win, reflectance(xp, dn, r["sat"]), use))
    n_obs = xp.zeros((h, w), dtype=xp.uint8)
    n_oli = xp.zeros((h, w), dtype=xp.uint8)
    days = xp.zeros((h, w), dtype=xp.int32)
    for p, t in taken.items():
        n_obs += t.astype(xp.uint8)
        if info[p][0]:
            n_oli += t.astype(xp.uint8)
        days += t.astype(xp.int32) * info[p][1]
    inside_np, n_np = _to_np(inside), _to_np(n_obs)
    day = np.rint(_to_np(days) / np.maximum(n_np, 1)).astype(np.uint8)       # 0 where there is no observation
    n_out = np.where(inside_np, n_np, NO_OBS).astype(np.uint8)
    extra = dict(n_obs_oli=np.where(inside_np, _to_np(n_oli), NO_OBS).astype(np.uint8),
                 obs_day=np.where(inside_np, day, NO_OBS).astype(np.uint8))
    if not obs:
        return None, n_out, px, extra
    out = {}
    with np.errstate(all="ignore"):
        for name in INDICES:
            d = INDEX_DEFS[name]
            stack = xp.full((len(passes), h, w), np.inf, dtype=xp.float32)
            for p, win, refl, use in obs:
                v = eval_index(d, refl, xp).astype(xp.float32)
                stack[(p,) + win] = xp.where(use & xp.isfinite(v), v, stack[(p,) + win])
            count = (stack != np.inf).sum(axis=0)
            packed = pack(xp, median_of(xp, stack, count), d["scale"])
            px["unstorable"] += int((count > 0).sum()) - int((packed != FILL).sum())
            out[name] = _to_np(packed)
            if name in SPREAD_INDICES:           # the stack is sorted now: smallest first, largest at count - 1
                top = xp.take_along_axis(stack, (xp.maximum(count, 1) - 1)[None], axis=0)[0]
                spread = xp.where(count > 0, top - stack[0], xp.float32(np.nan))
                out[f"{name}_range"] = _to_np(pack(xp, spread, d["scale"]))
    return out, n_out, px, extra


# ---------- check 1: the engine against plain, separately written 64-bit NumPy ----------
def _synthetic_month():
    """A made-up month: a 300 x 340 pixel frame with a hole in its mask and six scenes of two paths
    and two sensors, shifted by whole pixels, with fill, cloud, saturation and out-of-range values.
    Two pairs of scenes belong to one pass each and overlap, as consecutive rows of a path do."""
    rng = np.random.default_rng(11)
    h, w = 300, 340
    mask = np.ones((h, w), dtype=bool)
    mask[:40, :60] = False
    mask[200:230, 100:180] = False
    parts = []
    sat_ = lambda name, k: name if name in SATELLITES else SATELLITES[k % len(SATELLITES)]   # only chosen satellites
    for i, (sat, path, day, fr0, fc0, sh, sw) in enumerate((
            (sat_("LC08", 0), 30, 5, 0, 0, 170, 300), (sat_("LC08", 0), 30, 5, 150, 10, 150, 300),      # one pass, two rows
            (sat_("LE07", 1), 31, 12, 30, 20, 270, 320), (sat_("LC09", 2), 30, 13, 10, 60, 290, 250),
            (sat_("LT05", 3), 31, 20, 0, 0, 160, 340), (sat_("LT05", 3), 31, 20, 140, 5, 160, 330))):   # one pass, two rows
        mm = np.zeros((N_BANDS, sh, sw), dtype="<u2")
        u = rng.random((sh, sw))
        mm[0] = np.where(u < 0.08, 1, np.where(u < 0.35, 21824 | 8, 21824))     # fill, cloud, clear
        mm[1] = (rng.random((sh, sw)) < 0.01) * 2
        for k in range(2, 2 + N_REFL):
            mm[k] = rng.integers(7000, 21000, (sh, sw))
        if AEROSOL_SCREEN:                       # some hazy pixels, in the way each sensor stores them
            mm[N_BANDS - 1] = (rng.integers(1, 4, (sh, sw)) << 6) if SENSOR[sat] == "OLI" else \
                rng.integers(-50, 420, (sh, sw)).astype("<i2").view("<u2")
        r = dict(id=f"selftest{i}", sat=sat, path=path, t=f"2020-06-{day:02d}T17:00:{i:02d}.000Z")
        parts.append((r, dict(fr0=fr0, fc0=fc0, h=sh, w=sw, mm=mm, bands=scene_bands(r))))
    return parts, mask


def _reference_month(parts, mask):
    """The same month by a deliberately different route: 64-bit, one scene at a time onto a full
    frame, a pass put together pixel by pixel from its scenes, NumPy's own nanmedian. Returns
    ({index: float64 array}, n_obs)."""
    h, w = mask.shape
    groups = {}
    for r, part in parts:
        groups.setdefault((r["sat"], r["path"], r["t"][:10]), []).append((r, part))
    cube = {name: [] for name in INDICES}
    n_obs = np.zeros((h, w), dtype=int)
    with np.errstate(all="ignore"):
        for scenes in groups.values():
            seen = np.zeros((h, w), dtype=bool)
            value = {name: np.full((h, w), np.nan) for name in INDICES}
            for r, part in sorted(scenes, key=lambda x: x[0]["t"]):
                full = np.zeros((N_BANDS, h, w), dtype="i8")
                full[0] = 1
                full[:, part["fr0"]:part["fr0"] + part["h"], part["fc0"]:part["fc0"] + part["w"]] = part["mm"]
                use = mask & ((full[0] & 1) == 0) & ((full[0] & QA_PIXEL_REJECT) == 0)
                if REJECT_SATURATED:
                    use &= full[1] == 0
                if REQUIRE_VALID_RANGE:
                    use &= np.all((full[2:8] >= SR_VALID[0]) & (full[2:8] <= SR_VALID[1]), axis=0)
                if AEROSOL_SCREEN:
                    a = full[N_BANDS - 1]
                    use &= ~(((a // 64) == 3) if SENSOR[r["sat"]] == "OLI" else (np.where(a >= 32768, a - 65536, a) > OPACITY_MAX * 1000))
                use &= ~seen
                seen |= use
                letters = BAND_LETTERS[SENSOR[r["sat"]]]
                ref = reflectance(np, {L: full[part["bands"].index(letters[L])] for L in LETTERS}, r["sat"], np.float64)
                for name in INDICES:
                    v = np.asarray(eval_index(INDEX_DEFS[name], ref, np), dtype="f8")
                    value[name][use & np.isfinite(v)] = v[use & np.isfinite(v)]
            n_obs += seen
            for name in INDICES:
                cube[name].append(value[name])
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return {name: np.nanmedian(np.stack(v), axis=0) for name, v in cube.items()}, n_obs


def engine_self_test():
    """Three tests on the active engine. (1) Random reflectances through every formula, against
    64-bit NumPy. (2) The median, against NumPy's nanmedian, for 1 to 7 observations with gaps.
    (3) A made-up month through the whole chain (placing scenes on the frame, joining the scenes
    of a pass, pixel rule, maths, median, packing) against a separately written 64-bit version; on
    a GPU also against the CPU.
    A GPU that fails is dropped in favour of the CPU instead of stopping the run."""
    global XP, ENGINE
    rng = np.random.default_rng(7)
    ref = {L: rng.uniform(0.02, 0.6, 200_000) for L in LETTERS}
    for attempt in (0, 1):
        try:
            dev = {L: XP.asarray(v.astype(CDTYPE)) for L, v in ref.items()}
            worst = {}
            with np.errstate(all="ignore"):
                for name in INDICES:
                    want = eval_index(INDEX_DEFS[name], ref, np)
                    got = _to_np(eval_index(INDEX_DEFS[name], dev, XP)).astype("f8")
                    worst[name] = float(np.nanmedian(np.abs(got - want) / (1 + np.abs(want))))
            bad = {k: v for k, v in worst.items() if not v <= 1e-5}
            if bad:
                raise DataIntegrityError(f"formulas: median relative error above 1e-5: {bad}")
            for k in range(1, 8):
                vals = rng.normal(0, 1, (k, 5000)).astype("f4")
                vals[rng.random((k, 5000)) < 0.4] = np.nan
                vals[:, :50] = np.nan
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    want = np.nanmedian(vals.astype("f8"), axis=0).astype("f4")
                stack = XP.asarray(np.where(np.isnan(vals), np.inf, vals))
                got = _to_np(median_of(XP, stack, (stack != np.inf).sum(axis=0)))
                if not np.array_equal(got, want, equal_nan=True):
                    raise DataIntegrityError(f"median of {k} observations differs from NumPy's nanmedian")
            parts, mask = _synthetic_month()
            want, n_want = _reference_month(parts, mask)
            results = {"CPU": compose_strip(parts, mask, 0, mask.shape[0], 0, mask.shape[1], xp=np)}
            if XP is not np:
                results["GPU"] = compose_strip(parts, mask, 0, mask.shape[0], 0, mask.shape[1], xp=XP)
            off = 0.0
            for who, (arrays, n_obs, px, extra) in results.items():
                if not np.array_equal(n_obs, np.where(mask, n_want, NO_OBS)):
                    raise DataIntegrityError(f"{who}: n_obs differs from the reference")
                if px["out_of_range"] == 0 or px["clear"] == 0:
                    raise DataIntegrityError(f"{who}: the synthetic month gave an implausible result {px}")
                for name in INDICES:
                    q = np.rint(want[name] / INDEX_DEFS[name]["scale"])
                    q = np.where(np.isfinite(q) & (np.abs(q) <= 32767), q, FILL)
                    d = np.abs(arrays[name].astype("i8") - q.astype("i8"))
                    d = d[(arrays[name] != FILL) & (q != FILL)]
                    share = float((d > 1).mean()) if d.size else 0.0
                    if share > 1e-3 or float(((arrays[name] == FILL) != (q == FILL)).mean()) > 1e-3:
                        raise DataIntegrityError(f"{who}: {name} differs from the 64-bit reference by more than one "
                                                 f"storage step in {share:.2%} of the pixels")
                    off = max(off, float((d > 0).mean()) if d.size else 0.0)
            if XP is not np:
                for name in INDICES:
                    d = np.abs(results["GPU"][0][name].astype("i8") - results["CPU"][0][name].astype("i8"))
                    if float((d > 1).mean()) > 1e-4:
                        raise DataIntegrityError(f"{name} differs between GPU and CPU by more than one storage step")
            log.info("Engine self-test OK on %s: formulas within %.1e of 64-bit; median equal to NumPy's nanmedian; "
                     "a synthetic month through the whole chain equals a separate 64-bit version to within one "
                     "storage step (%.2f %% of values differ by that step)%s", ENGINE, max(worst.values()),
                     100 * off, "" if XP is np else "; GPU = CPU")
            return worst
        except Exception as e:
            if XP is np or attempt:
                raise
            log.warning("GPU engine failed its self-test (%s) - continuing on the CPU", e)
            XP, ENGINE = np, f"CPU (NumPy {np.__version__}, GPU self-test failed)"


# ---------- check 2: the same pixels through eemont on Earth Engine's servers ----------
def eemont_cross_check(r):
    """A small window of scene r, computed twice in 64-bit: by eemont on Earth Engine (scaleAndOffset
    + spectralIndices) and here from the downloaded integers. The two must agree to rounding, which
    proves the formulas, band letters, scale and offset are eemont's. With HARMONIZE the same linear
    adjustment is applied on both sides."""
    n = EEMONT_CHECK_PX
    row0, col0 = (r["h"] - n) // 2, (r["w"] - n) // 2
    letters = BAND_LETTERS[SENSOR[r["sat"]]]
    bands = sorted(set(letters.values()))
    g = _grid(r, row0, col0, n, n)
    raw = with_retries(lambda: ee.data.getPixels(dict(assetId=r["name"], bandIds=bands, grid=g,
                                                      fileFormat="NUMPY_NDARRAY")), "cross-check pixels")
    present = np.all([raw[b] != 0 for b in bands], axis=0)
    if not present.any():
        log.warning("Cross-check window of %s holds no data - skipped", r["id"])
        return None
    dn = {L: raw[letters[L]][present] for L in LETTERS}
    scaled = ee.Image(f"{collection_id(r['sat'])}/{r['id']}").scaleAndOffset()                  # eemont
    if HARMONIZE and SENSOR[r["sat"]] == "TM":
        names_ = [letters[L] for L in LETTERS]
        adjusted = (scaled.select(names_).multiply(ee.Image.constant([HARMONIZATION[L][0] for L in LETTERS]))
                    .add(ee.Image.constant([HARMONIZATION[L][1] for L in LETTERS])).rename(names_))
        scaled = scaled.addBands(adjusted, None, True)
    server, by_params = {}, {}
    for name in INDICES:
        by_params.setdefault(json.dumps(INDEX_DEFS[name]["params"], sort_keys=True), []).append(name)
    for names in by_params.values():
        kw = dict(INDEX_PARAMS, **INDEX_DEFS[names[0]]["params"])
        img = scaled.spectralIndices(names, G=kw["g"], C1=kw["C1"], C2=kw["C2"], L=kw["L"]).select(names)  # eemont
        got = with_retries(lambda: ee.data.computePixels(dict(expression=img.toDouble(), grid=g,
                                                              fileFormat="NUMPY_NDARRAY")), "cross-check eemont")
        for name in names:
            server[name] = got[name][present].astype("f8")
    r64 = reflectance(np, dn, r["sat"], np.float64)
    worst = 0.0
    with np.errstate(all="ignore"):
        for name in INDICES:
            d = INDEX_DEFS[name]
            cat = dict(d, code=_compile_formula(d["catalogue_formula"])[0])      # eemont's own formula text
            ok = np.isfinite(server[name])
            e64 = np.abs(eval_index(cat, r64, np) - server[name])[ok] / (1 + np.abs(server[name][ok]))
            if e64.size and e64.max() > 1e-9:
                raise DataIntegrityError(f"{name} on {r['id']}: the local 64-bit result differs from eemont on Earth "
                                         f"Engine by {e64.max():.2e} (relative). Formula, band letters or scaling "
                                         "do not match eemont.")
            worst = max(worst, float(e64.max()) if e64.size else 0.0)
    log.info("eemont cross-check OK on %s (%s, %d px): 64-bit agreement %.1e", r["id"], SAT_NAME[r["sat"]],
             int(present.sum()), worst)
    return dict(scene=r["id"], agreement_64bit=worst, harmonized=bool(HARMONIZE and SENSOR[r["sat"]] == "TM"))

In [ ]:
# ═══════════════ CELL 8: the yearly CF NetCDF file: writer, state, read-back check, audit ═══════════════
SETTINGS_SIGNATURE = json.dumps(dict(
    indices=INDICES, formulas={k: INDEX_DEFS[k]["formula"] for k in INDICES},
    params={k: INDEX_DEFS[k]["params"] for k in INDICES}, scales={k: INDEX_DEFS[k]["scale"] for k in INDICES},
    qa_reject=QA_PIXEL_REJECT, saturated=REJECT_SATURATED, valid_range=REQUIRE_VALID_RANGE,
    harmonize=HARMONIZE, tier=TIER, pixel=PIXEL, composite="monthly median",
    aerosol=[AEROSOL_SCREEN, OPACITY_MAX], landsat7_last_day=LANDSAT7_LAST_DAY, spread=SPREAD_INDICES), sort_keys=True)
DRIVE_RETRY_S = 20                       # pause before a file on Drive is read again
STRIP = max(1, STRIP_ROWS // CHUNK_YX) * CHUNK_YX      # strips are whole rows of chunks, so each chunk is written once


TILE_RE = re.compile(r"z\d\de\d+n\d+")


def unit_dir(key, year):
    """Folder of one tile's file of one year: <output folder>/<year>/<tile>/"""
    return os.path.join(DRIVE_OUT, str(year), key)


def unit_name(key, year):
    return f"{PRODUCT_TAG}_{key}_{year}_{ROI_LABEL}.nc"


def _cf_attr(v):
    if isinstance(v, (str, int, float, np.number, np.ndarray)):
        return v
    if isinstance(v, (list, tuple)):
        return np.asarray(v)
    return str(v)


def pixel_rule_text():
    bits = {1: "dilated cloud", 2: "cirrus", 3: "cloud", 4: "cloud shadow", 5: "snow"}
    rule = "fill, " + ", ".join(v for b, v in bits.items() if QA_PIXEL_REJECT >> b & 1) + " (QA_PIXEL)"
    if REJECT_SATURATED:
        rule += "; any band saturated (QA_RADSAT)"
    if REQUIRE_VALID_RANGE:
        rule += f"; a reflective band outside USGS's valid range {SR_VALID[0]}-{SR_VALID[1]}"
    if AEROSOL_SCREEN:
        rule += (f"; hazy air: aerosol level high (SR_QA_AEROSOL, Landsat 8/9) or atmospheric opacity above "
                 f"{OPACITY_MAX} (SR_ATMOS_OPACITY, Landsat 5/7)")
    return rule


def harmonisation_text():
    if not HARMONIZE:
        return ("none: reflectance of every sensor is used as distributed in Landsat Collection 2 Level-2. Indices "
                "from TM/ETM+ (Landsat 5, 7) and OLI (Landsat 8, 9) can differ slightly because the bands differ.")
    return ("Landsat 5 TM and Landsat 7 ETM+ surface reflectance adjusted to Landsat 8 OLI before the indices were "
            "computed: OLI = intercept + slope x ETM+, ordinary least squares, Roy et al. 2016, Table 2 "
            "(https://doi.org/10.1016/j.rse.2015.12.024); " + ", ".join(
                f"{L}: {HARMONIZATION[L][1]:+.4f} + {HARMONIZATION[L][0]:.4f} x" for L in LETTERS) +
            ". These coefficients were derived before Collection 2 and for ETM+; their use here for Collection 2 "
            "and for TM is an assumption.")


def create_year_nc(path, P, year, mask):
    """Create the empty CF file of one tile and year on the tile's fixed 30 m frame.
    Written under a temporary name and renamed when complete."""
    F, key = P["F"], P["key"]
    crs = pyproj.CRS.from_user_input(F["crs"])
    zkw = dict(zlib=COMPRESS_LEVEL > 0, complevel=max(COMPRESS_LEVEL, 1), shuffle=True)
    cy, cx = min(CHUNK_YX, F["h"]), min(CHUNK_YX, F["w"])
    tmp = path + ".new"
    ds = nc4.Dataset(tmp, "w", format="NETCDF4")
    try:
        ds.createDimension("time", None)
        ds.createDimension("y", F["h"])
        ds.createDimension("x", F["w"])
        ds.createDimension("nv", 2)
        ds.createDimension("satellite", len(SATELLITES))
        tv = ds.createVariable("time", "f8", ("time",), chunksizes=(12,))
        tv.units, tv.calendar, tv.standard_name, tv.axis = TIME_UNITS, TIME_CALENDAR, "time", "T"
        tv.long_name, tv.bounds = "first day of the month (UTC)", "time_bnds"
        ds.createVariable("time_bnds", "f8", ("time", "nv"), chunksizes=(12, 2))
        yv = ds.createVariable("y", "f8", ("y",))
        yv.standard_name, yv.units, yv.axis = "projection_y_coordinate", "m", "Y"
        yv.long_name = "northing of pixel centre"
        yv[:] = F["y0"] - (np.arange(F["h"]) + 0.5) * PIXEL
        xv = ds.createVariable("x", "f8", ("x",))
        xv.standard_name, xv.units, xv.axis = "projection_x_coordinate", "m", "X"
        xv.long_name = "easting of pixel centre"
        xv[:] = F["x0"] + (np.arange(F["w"]) + 0.5) * PIXEL
        gm = ds.createVariable("crs", "i4")
        for k, v in crs.to_cf().items():
            gm.setncattr(k, _cf_attr(v))
        gm.spatial_ref = crs.to_wkt()
        gm.GeoTransform = f"{F['x0']} {PIXEL} 0.0 {F['y0']} 0.0 {-PIXEL}"       # GDAL order
        gm.long_name = f"Landsat's own 30 m grid of UTM zone {int(key[1:3])} ({F['crs']}), tile {key}"

        mv = ds.createVariable("roi_mask", "i1", ("y", "x"), chunksizes=(cy, cx), **zkw)
        mv.long_name = f"1 where the pixel centre lies inside the {ROI_LABEL} aquifer polygon"
        mv.flag_values = np.array([0, 1], dtype="i1")
        mv.flag_meanings = "outside_roi inside_roi"
        mv.grid_mapping = "crs"
        for r0 in range(0, F["h"], 2048):
            mv[r0:r0 + 2048, :] = mask[r0:r0 + 2048].astype("i1")

        for name in INDICES:
            d = INDEX_DEFS[name]
            v = ds.createVariable(name, "i2", ("time", "y", "x"), fill_value=np.int16(FILL), chunksizes=(1, cy, cx), **zkw)
            v.long_name = d["long_name"]
            v.units = "1"
            v.scale_factor = np.float32(d["scale"])       # a 32-bit factor: readers unpack to 32-bit floats
            v.grid_mapping = "crs"
            v.cell_methods = "time: median (of the usable observations of the month, one per satellite pass)"
            v.ancillary_variables = "n_obs"
            v.formula = d["formula"]
            v.formula_source = d["origin"]
            v.formula_bands = "B: blue, G: green, R: red, N: near infrared, S1: shortwave infrared 1, S2: shortwave infrared 2"
            if d["params"]:
                v.formula_parameters = json.dumps(d["params"], sort_keys=True)
            if d["kernels"]:
                v.formula_kernel = "k(a, b) = exp(-(a - b)^2 / (2 sigma^2)), sigma = 0.5 (a + b)  [eemont default, RBF]"
            v.references = d["reference"]
            v.application_domain = d["domain"]
            v.comment = ("Computed per 30 m pixel and per scene from Landsat Collection 2 Level-2 surface reflectance; "
                         "the value is the median over the pixel's usable observations of the month, from every "
                         "path/row of this UTM zone that covers it (the mean of the two middle ones when their number "
                         "is even). Observations left out: " + pixel_rule_text() + ". _FillValue: outside the "
                         "polygon, no usable observation, or a value this scale cannot hold.")
        nv = ds.createVariable("n_obs", "u1", ("time", "y", "x"), fill_value=np.uint8(NO_OBS), chunksizes=(1, cy, cx), **zkw)
        nv.long_name, nv.units, nv.grid_mapping = "number of usable observations of the pixel in the month", "1", "crs"
        nv.comment = ("One observation = one satellite pass in which the pixel is usable; two scenes of one pass that "
                      "overlap count once. 0: inside the polygon but no usable observation. _FillValue marks pixels "
                      "outside the polygon.")

        for name in RANGE_VARS:
            d = INDEX_DEFS[name[:-6]]
            v = ds.createVariable(name, "i2", ("time", "y", "x"), fill_value=np.int16(FILL), chunksizes=(1, cy, cx), **zkw)
            v.long_name = f"range of {name[:-6]} over the usable observations of the month (largest minus smallest)"
            v.units, v.grid_mapping, v.scale_factor = "1", "crs", np.float32(d["scale"])
            v.cell_methods = "time: range"
            v.comment = "0 when the month has a single observation. Large values point to change or leftover cloud within the month."
        ov = ds.createVariable("n_obs_oli", "u1", ("time", "y", "x"), fill_value=np.uint8(NO_OBS), chunksizes=(1, cy, cx), **zkw)
        ov.long_name, ov.units, ov.grid_mapping = "number of the month's observations that come from Landsat 8 or 9 (OLI)", "1", "crs"
        ov.comment = "n_obs minus this is the number from Landsat 5 or 7 (TM, ETM+)."
        dv = ds.createVariable("obs_day", "u1", ("time", "y", "x"), fill_value=np.uint8(NO_OBS), chunksizes=(1, cy, cx), **zkw)
        dv.long_name, dv.units, dv.grid_mapping = "mean day of the month of the pixel's usable observations, rounded", "1", "crs"
        dv.comment = "0: no usable observation. Days are UTC acquisition days."

        sv = ds.createVariable("satellite", "i1", ("satellite",))
        sv.long_name = "Landsat satellite number"
        sv[:] = np.array([SAT_NUMBER[s] for s in SATELLITES], dtype="i1")
        ds.createVariable("n_scenes", "i2", ("time",), chunksizes=(12,)).long_name = "scenes read for the month"
        ds.createVariable("n_scenes_by_satellite", "i2", ("time", "satellite"), chunksizes=(12, len(SATELLITES))).long_name = \
            "scenes read for the month, per satellite"
        ds.createVariable("source_scenes", str, ("time",)).long_name = "Earth Engine ids of the scenes read for the month"

        ds.Conventions = "CF-1.9"
        ds.title = (f"Landsat Collection 2 Level-2 vegetation and water indices, {ROI_LABEL} aquifer, UTM zone "
                    f"{int(key[1:3])} tile {key}, {year}: monthly medians at Landsat's own 30 m resolution")
        ds.summary = ("One record per month. Each index is computed per 30 m pixel for every scene of the month; the "
                      "record holds the median of the pixel's usable observations, from every path/row of this UTM "
                      "zone that covers it, and their number. Scenes are placed on the zone's grid by whole-pixel "
                      "shifts: nothing is resampled or gap-filled. Scenes of the neighbouring UTM zone are not used.")
        ds.institution = "U.S. Geological Survey (source data); packaged by the MSU groundwater project"
        ds.source = ("USGS Landsat Collection 2 Level-2 Tier 1 surface reflectance, read as stored integers from "
                     "Google Earth Engine (" + ", ".join(collection_id(s) for s in SATELLITES) + "); index formulas, "
                     f"scale and offset from eemont {eemont.__version__}")
        ds.references = ("Landsat Collection 2 Level-2 Science Products: " + ", ".join(
            f"https://doi.org/{d}" for d in sorted({SAT_DOI[s] for s in SATELLITES})) + "; eemont: "
            "https://doi.org/10.21105/joss.03168; Awesome Spectral Indices: https://doi.org/10.1038/s41597-023-02096-0")
        ds.history = f"{iso(utcnow())} created by Landsat_Indices_30m_Monthly_Ogallala.ipynb"
        ds.tile, ds.utm_zone, ds.year, ds.roi = key, np.int32(int(key[1:3])), np.int32(year), ROI_LABEL
        ds.indices = " ".join(INDICES)
        ds.observations_left_out = pixel_rule_text()
        ds.sensor_harmonisation = harmonisation_text()
        ds.landsat7_last_day = LANDSAT7_LAST_DAY or "none (every Landsat 7 scene is used)"
        ds.frame = json.dumps(dict(crs=F["crs"], x0=F["x0"], y0=F["y0"], w=F["w"], h=F["h"]))
        ds.settings_signature = SETTINGS_SIGNATURE
        ds.processing_status = "in_progress"
        ds.months = ""
        ds.committed_records = np.int32(0)
    finally:
        ds.close()
    os.replace(tmp, path)


def read_state(path, tries=1):
    """(state, None) of a yearly file, or (None, why) when it cannot be used: unreadable, a record
    count that disagrees with its last commit, or months out of order. From Drive it is tried
    several times: a Drive that is slow or only just mounted must never be mistaken for a damaged file."""
    why = None
    for attempt in range(max(1, tries)):
        if attempt:
            time.sleep(DRIVE_RETRY_S * attempt)
        try:
            with nc4.Dataset(path, "r") as ds:
                n = len(ds.variables["time"])
                months = [int(m) for m in str(getattr(ds, "months", "")).split()]
                if int(getattr(ds, "committed_records", -1)) != n or len(months) != n:
                    why = (f"holds {n} records but its last commit says {getattr(ds, 'committed_records', '?')} "
                           "(an interrupted write)")
                elif months != sorted(set(months)):
                    why = "has its months out of order"
                else:
                    return dict(n=n, months=months, status=str(getattr(ds, "processing_status", "in_progress")),
                                signature=str(getattr(ds, "settings_signature", "")),
                                satellites=[int(v) for v in ds.variables["satellite"][:]],
                                frame=json.loads(str(getattr(ds, "frame", "{}"))), size=os.path.getsize(path)), None
        except Exception as e:
            why = f"cannot be read ({type(e).__name__}: {str(e)[:160]})"
        log.warning("%s %s (attempt %d of %d)", os.path.basename(path), why, attempt + 1, max(1, tries))
    return None, why


class YearWriter:
    """Single writer of one yearly file on scratch. A month is written strip by strip and counted
    only when finish_month() has stored it in the file's own list of months."""

    def __init__(self, path):
        self.path, self.ds = path, None
        self.open()

    def open(self):
        self.ds = nc4.Dataset(self.path, "a")
        self.ds.set_auto_maskandscale(False)
        self.n = len(self.ds.variables["time"])
        self.months = [int(m) for m in str(self.ds.months).split()]

    def write_strip(self, r0, r1, c0, c1, arrays, n_obs, extra):
        v = self.ds.variables
        v["n_obs"][self.n, r0:r1, c0:c1] = n_obs
        for name, a in extra.items():
            v[name][self.n, r0:r1, c0:c1] = a
        for name, a in (arrays or {}).items():
            v[name][self.n, r0:r1, c0:c1] = a

    def _in_order(self, month):
        if self.months and month <= self.months[-1]:
            raise DataIntegrityError(f"{os.path.basename(self.path)}: month {month} after month {self.months[-1]}")

    def finish_month(self, year, month, scenes, status=None):
        self._in_order(month)
        ds, v, i = self.ds, self.ds.variables, self.n
        start, end = month_bounds(year, month)
        v["time"][i] = (start - EPOCH).days
        v["time_bnds"][i, :] = [(start - EPOCH).days, (end - EPOCH).days]
        v["n_scenes"][i] = len(scenes)
        v["n_scenes_by_satellite"][i, :] = [sum(1 for r in scenes if r["sat"] == s) for s in SATELLITES]
        v["source_scenes"][i] = " ".join(r["id"] for r in scenes)
        self.n, self.months = i + 1, self.months + [month]
        self.commit(status)

    def take_month(self, old, t_old, month):
        """Count a month whose arrays were copied from another file of this tile and year; its
        record (time, scenes read) is copied from there as well."""
        self._in_order(month)
        v, o, i = self.ds.variables, old.variables, self.n
        for name in ("time", "time_bnds", "n_scenes", "n_scenes_by_satellite", "source_scenes"):
            v[name][i] = o[name][t_old]
        self.n, self.months = i + 1, self.months + [month]
        self.commit()

    def commit(self, status=None):
        ds = self.ds
        ds.months = " ".join(str(m) for m in self.months)
        ds.committed_records = np.int32(self.n)
        if status:
            ds.processing_status = status
        if self.n:
            ds.time_coverage_start = iso(month_bounds(int(ds.year), self.months[0])[0])
            ds.time_coverage_end = iso(month_bounds(int(ds.year), self.months[-1])[1])
        ds.sync()

    def close(self):
        if self.ds is not None:
            self.ds.close()
            self.ds = None


def strip_windows(mask):
    """The parts of a frame that are written, as (r0, r1, c0, c1): strips of rows, their columns cut
    to the chunks that hold aquifer pixels. Rows without any aquifer pixel are left out, so that
    their chunks stay unwritten."""
    ny, nx = mask.shape
    for r0 in range(0, ny, STRIP):
        r1 = min(ny, r0 + STRIP)
        cols = np.flatnonzero(mask[r0:r1].any(axis=0))
        if cols.size:
            yield r0, r1, int(cols[0]) // CHUNK_YX * CHUNK_YX, min(nx, -(-(int(cols[-1]) + 1) // CHUNK_YX) * CHUNK_YX)


def copy_month(old, t_old, month, writer, mask):
    """One month of another file of the same tile and year into the file being written, exactly as
    it is stored there: the 16-bit integers are copied, never unpacked or computed again. Returns
    the checksums of what was read, so that the closed file can be compared with it."""
    crcs = {}
    for r0, r1, c0, c1 in strip_windows(mask):
        for name in COUNT_VARS + INDICES + RANGE_VARS:
            a = np.ascontiguousarray(old.variables[name][t_old, r0:r1, c0:c1])
            if name in COUNT_VARS or np.count_nonzero(a != FILL):    # an array of fill values stays unwritten
                writer.ds.variables[name][writer.n, r0:r1, c0:c1] = a
            crcs[(name, r0, r1, c0, c1)] = zlib.crc32(a)
    writer.take_month(old, t_old, month)
    return crcs


def readback_month(path, t, crcs):
    """Read one month back from the closed file, strip by strip, and compare each array's checksum
    with the one taken when it was computed."""
    with nc4.Dataset(path, "r") as ds:
        ds.set_auto_maskandscale(False)
        for (name, r0, r1, c0, c1), want in crcs.items():
            got = np.ascontiguousarray(ds.variables[name][t, r0:r1, c0:c1])
            if zlib.crc32(got) != want:
                raise DataIntegrityError(f"{os.path.basename(path)} record {t} {name} rows {r0}-{r1}: the file "
                                         "differs from what was computed")
    return len(crcs)


def audit_nc(path, mask=None):
    """Read-only audit of a yearly file. Returns (ok, problems, info). With the mask, three strips
    of the first and last month are also checked against it."""
    problems, info = [], {}
    try:
        with nc4.Dataset(path, "r") as ds:
            ds.set_auto_maskandscale(False)
            if not str(getattr(ds, "Conventions", "")).startswith("CF-"):
                problems.append("no CF Conventions attribute")
            names = str(getattr(ds, "indices", "")).split()
            need = ["time", "time_bnds", "y", "x", "crs", "roi_mask", "n_obs", "n_scenes", "source_scenes"] + names
            missing = [v for v in need if v not in ds.variables]
            if missing or not names:
                return False, [f"missing variables {missing}"], info
            n = len(ds.variables["time"])
            months = [int(m) for m in str(getattr(ds, "months", "")).split()]
            t = np.asarray(ds.variables["time"][:], dtype="f8")
            if int(getattr(ds, "committed_records", -1)) != n or len(months) != n:
                problems.append("record count differs from the committed months")
            if months != sorted(set(months)) or (n > 1 and not np.all(np.diff(t) > 0)):
                problems.append("months not strictly increasing")
            year = int(getattr(ds, "year", 0))
            if n and [(month_bounds(year, m)[0] - EPOCH).days for m in months] != [int(v) for v in t]:
                problems.append("time values do not match the months")
            for name in names + ["n_obs"]:
                v = ds.variables[name]
                if v.dimensions != ("time", "y", "x") or getattr(v, "grid_mapping", None) != "crs":
                    problems.append(f"{name}: unexpected dimensions or no grid_mapping")
            ny, nx = len(ds.dimensions["y"]), len(ds.dimensions["x"])
            if mask is not None and n:
                if mask.shape != (ny, nx):
                    problems.append("the file is not on the tile's frame")
                else:
                    for i in sorted({0, n - 1}):
                        for r0 in sorted({0, (ny // 2 // STRIP) * STRIP, ((ny - 1) // STRIP) * STRIP}):
                            m = mask[r0:r0 + STRIP]
                            c = np.asarray(ds.variables["n_obs"][i, r0:r0 + STRIP, :])
                            a = np.asarray(ds.variables[names[0]][i, r0:r0 + STRIP, :])
                            if np.count_nonzero(c[~m] != NO_OBS) or np.count_nonzero(a[~m] != FILL):
                                problems.append(f"record {i} rows {r0}+: values outside the polygon")
                            if np.count_nonzero(c[m] == NO_OBS):
                                problems.append(f"record {i} rows {r0}+: no count inside the polygon")
                            if np.count_nonzero((a != FILL) & (c == 0)):
                                problems.append(f"record {i} rows {r0}+: a value where no observation was counted")
            info.update(records=n, months=months, status=str(getattr(ds, "processing_status", "?")),
                        shape=f"{ny}x{nx}", size=os.path.getsize(path))
    except Exception as e:
        problems.append(f"cannot be opened/read: {type(e).__name__}: {e}")
    return (not problems), problems, info

In [ ]:
# ═══════════════ CELL 9: year by year, tile by tile: one yearly file per unit; run ═══════════════
# How a run keeps its place. The unit of work is one tile and one year: some dozens of scenes, a few
# minutes. Its file is built on scratch, checked, and only then copied to Drive under its final name, in one
# verified step. Nothing on Drive is ever half written, appended to or compressed twice. A runtime that dies
# loses the unit it was working on and nothing else; the next run starts with the first file that is missing.
#
# Which months a run makes is set by START_MONTH and STOP_MONTH. A yearly file holds the months made so far.
# When a later run wants more months of that year, the file on Drive is fetched and the new months are
# added to it; a month that belongs before one already there is written into a new file, into which the
# months on Drive are copied as they are stored. Either way a month that is on Drive is never computed again.
SIG_ID = hashlib.md5(SETTINGS_SIGNATURE.encode()).hexdigest()[:12]
LAST = {}                       # the last file that reached Drive (for the message printed at a stop)


def frame_id(F):
    return [F["crs"], F["x0"], F["y0"], F["w"], F["h"]]


def load_ledger():
    """Yearly files already on Drive, as noted when each was uploaded (the last note of a file counts)."""
    led, path = {}, os.path.join(DRIVE_OUT, LEDGER_NAME)
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                try:
                    e = json.loads(line)
                    led[e["file"]] = e
                except (ValueError, KeyError):
                    pass
    return led


def save_to_drive(src, dst, what):
    """Verified copy to Drive; a Drive that will not take the file stops the run cleanly."""
    try:
        robust_drive_copy(src, dst)
    except PipelineStop:
        raise
    except Exception as e:
        raise PipelineStop(f"{what} could not be saved to Drive ({e}). Everything saved before it is still "
                           "there - re-run to resume.")


def adopt_old_layout():
    """Earlier versions of this notebook kept the files as <tile>/<file>. They are moved to
    <year>/<tile>/<file>: a rename on Drive, nothing is copied, changed or read. A file whose new
    place is already taken is left where it is."""
    moved, left = 0, []
    for d in sorted(glob.glob(os.path.join(DRIVE_OUT, "z*e*n*"))):
        key = os.path.basename(d)
        if not (os.path.isdir(d) and TILE_RE.fullmatch(key)):
            continue
        for p in sorted(glob.glob(os.path.join(d, f"{PRODUCT_TAG}_{key}_*_{ROI_LABEL}.nc"))):
            m = re.fullmatch(rf"{PRODUCT_TAG}_{key}_(\d{{4}})_{ROI_LABEL}\.nc", os.path.basename(p))
            if not m:
                continue
            dst = os.path.join(unit_dir(key, int(m.group(1))), os.path.basename(p))
            if os.path.exists(dst):
                left.append(p)
                continue
            try:
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                os.rename(p, dst)
                moved += 1
            except FileNotFoundError:
                pass                             # another session moved it a moment ago
            except OSError as e:
                raise PipelineStop(f"{os.path.basename(p)} could not be moved to {os.path.dirname(dst)} ({e}). "
                                   "Nothing is lost - run again.")
        try:
            os.rmdir(d)                          # goes only when it is empty
        except OSError:
            pass
    if moved:
        log.info("%d file(s) of the earlier folder layout <tile>/ moved to <year>/<tile>/ (renamed on Drive, not "
                 "copied)", moved)
    for p in left:
        log.warning("%s stays in the earlier layout: there is a file of that name in the new place already", p)


class Progress:
    def __init__(self, total):
        self.total, self.done, self.run, self.t0 = total, 0, 0, time.time()

    def add(self, n, resumed=False):
        self.done += n
        if not resumed:
            self.run += n

    def line(self):
        rate = self.run / max(time.time() - self.t0, 1e-9)
        eta = (self.total - self.done) / rate / 3600 if rate > 0 else float("nan")
        return (f"{self.done}/{self.total} scenes ({100 * self.done / max(1, self.total):.1f} %), "
                f"{rate * 60:.1f}/min, ETA {eta:.1f} h")


def unit_plan(P, year, ledger):
    """What one tile-year needs: nothing (its file on Drive holds every month this run wants), a
    new file, or more months for the file that is there ("append" when they all come after its
    last month, "merge" when one belongs before it). A file that cannot be read stops the run: it
    is never replaced or redone because of that."""
    key, name = P["key"], unit_name(P["key"], year)
    drive_nc = os.path.join(unit_dir(key, year), name)
    want = wanted_months(year)
    if not want:
        return dict(action="skip", months=[])
    if not os.path.exists(drive_nc):
        if name in ledger:
            log.warning("%s was uploaded earlier but is not on Drive now - it is made again", name)
        return dict(action="new", todo=want, have=[], drive_nc=drive_nc)
    led = ledger.get(name)
    if led and led.get("size") == os.path.getsize(drive_nc) and led.get("settings") == SIG_ID \
            and led.get("frame") == frame_id(P["F"]):
        have = [int(m) for m in led["months"]]
    else:
        st, why = read_state(drive_nc, DRIVE_READ_TRIES)
        if st is None:
            raise PipelineStop(
                f"{name} is on Drive but {why}. Nothing was changed or deleted. Run again in a few minutes (Drive "
                "can be slow just after it is mounted). If the file really is damaged, move it out of the folder "
                "yourself and run again: only that tile-year is then made again.")
        if st["signature"] != SETTINGS_SIGNATURE:
            raise PipelineStop(f"{name} was made with other settings (indices, formulas, scales, pixel rule or "
                               "harmonisation). Restore those settings in Cell 2, or use an empty output folder.")
        if [st["frame"].get(k) for k in ("crs", "x0", "y0", "w", "h")] != frame_id(P["F"]):
            raise PipelineStop(f"{name} is not on the frame stored in {FRAMES_NAME}. Nothing was changed.")
        have = st["months"]
        ledger[name] = dict(file=name, tile=key, year=year, months=have, size=st["size"], settings=SIG_ID,
                            frame=frame_id(P["F"]), note="found on Drive")
        log_events(LEDGER_NAME, [ledger[name]])
    todo = [m for m in want if m not in have]
    if not todo:
        return dict(action="skip", months=have)
    action = "append" if not have or min(todo) > max(have) else "merge"
    return dict(action=action, todo=todo, have=have, drive_nc=drive_nc)


def compose_month(parts, mask, writer, keep_checksums):
    """One month onto the yearly file, strip by strip. Returns (checksums or None, counters)."""
    opened = [(r, None if part is None else dict(part, mm=np.load(part["path"], mmap_mode="r"))) for r, part in parts]
    crcs = {} if keep_checksums else None
    px = dict(clear=0, out_of_range=0, unstorable=0, hazy=0, covered=0)
    try:
        for r0, r1, c0, c1 in strip_windows(mask):
            arrays, n_obs, st, extra = compose_strip(opened, mask[r0:r1, c0:c1], r0, r1, c0, c1)
            writer.write_strip(r0, r1, c0, c1, arrays, n_obs, extra)
            for k in ("clear", "out_of_range", "unstorable", "hazy"):
                px[k] += st[k]
            px["covered"] += int(np.count_nonzero((n_obs > 0) & (n_obs != NO_OBS)))
            if crcs is not None:
                for name, a in dict(extra, n_obs=n_obs, **(arrays or {})).items():
                    crcs[(name, r0, r1, c0, c1)] = zlib.crc32(np.ascontiguousarray(a))
    finally:
        del opened
    return crcs, px


def process_unit(P, year, plan, prog):
    """One tile and year: fetch a month (the next one is fetched while this one is computed and
    written), compose it, repeat; then check the closed file and copy it to Drive. Months that are
    on Drive already are kept as they are: the file is added to ("append") or, when a new month
    belongs before one of them, they are copied as stored into a new file ("merge")."""
    key, F = P["key"], P["F"]
    name, drive_nc = unit_name(key, year), plan["drive_nc"]
    tag = f"{key}_{year}"
    work, raw = os.path.join(NC_DIR, tag), os.path.join(RAW_DIR, tag)
    for d in (work, raw):
        shutil.rmtree(d, ignore_errors=True)     # scratch left by an earlier run is never used
    os.makedirs(work, exist_ok=True)
    local, t_start, months = os.path.join(work, name), time.time(), plan["todo"]
    kept = plan["have"] if plan["action"] == "merge" else []      # months copied from the file on Drive
    mask = frame_mask(key, F)
    scenes_of = lambda m: P["months"].get((year, m), [])
    stop, pool, writer, old = threading.Event(), ThreadPoolExecutor(1), None, None
    fetch = lambda m: fetch_month(f"{tag}/m{m:02d}", scenes_of(m), F, mask, stop)
    check = set(months) if TEST_MODE else set(months[:READBACK_MONTHS])
    counts, checks = dict(scenes=0, failed=0), {}
    try:
        try:
            on_drive = os.path.getsize(drive_nc) if plan["action"] != "new" else 0
            ensure_disk_free(f"{tag} file", extra_gb=(F["n_roi"] * 26 * (len(months) + len(kept)) + on_drive) / 1024 ** 3)
            header = None
            if plan["action"] != "new":
                fetched = local if plan["action"] == "append" else os.path.join(work, "on_drive.nc")
                restore_from_drive(drive_nc, fetched)
                st, why = read_state(fetched)
                if st is None or st["months"] != plan["have"]:
                    raise PipelineStop(f"The copy of {name} fetched from Drive is not the file that was read a "
                                       f"moment ago ({why or 'it has changed'}). Nothing was changed - run again.")
                if st["signature"] != SETTINGS_SIGNATURE:
                    raise PipelineStop(f"{name} was made with other settings (indices, formulas, scales, pixel rule or "
                                       "harmonisation). Restore those settings in Cell 2. Nothing was changed.")
                if st["satellites"] != [SAT_NUMBER[s] for s in SATELLITES]:
                    raise PipelineStop(f"{name} was made with another SATELLITES list (Landsat {st['satellites']}), so "
                                       "months made with this one cannot be added to it. Restore that list in Cell 2, "
                                       "or use an empty output folder. Nothing was changed.")
            if plan["action"] != "append":
                create_year_nc(local, P, year, mask)
            if plan["action"] == "new":
                header = os.path.getsize(local)
            writer = YearWriter(local)
            if kept:
                old = nc4.Dataset(fetched, "r")
                old.set_auto_maskandscale(False)
                if not np.array_equal(np.asarray(old.variables["roi_mask"][:]).astype(bool), mask):
                    raise PipelineStop(f"{name} on Drive was made with another aquifer mask than this run has. "
                                       "Nothing was changed.")
            fut, i = pool.submit(fetch, months[0]), 0
            for m in sorted(months + kept):
                if m in kept:                    # on Drive already: copied as stored, and compared after closing
                    t_c, t = time.time(), writer.n
                    checks[t] = copy_month(old, plan["have"].index(m), m, writer, mask)
                    TIMES["months copied"] = TIMES.get("months copied", 0.0) + time.time() - t_c
                    log.info("[%s-%02d] copied as stored from the file on Drive", tag, m)
                    continue
                parts, permanent = fut.result()
                i += 1
                fut = pool.submit(fetch, months[i]) if i < len(months) else None
                if permanent:
                    counts["failed"] += len(permanent)
                    log_events(LOG_FAILED, [dict(scene=r["id"], tile=key, reason=why) for r, why in permanent])
                    for r, why in permanent:
                        log.warning("[%s] permanently unavailable: %s (%s)", tag, r["id"], why[:160])
                t_c, t = time.time(), writer.n
                crcs, px = compose_month(parts, mask, writer, m in check)
                used = [r for r, part in parts if part is not None]
                writer.finish_month(year, m, used)
                TIMES["index maths + write"] = TIMES.get("index maths + write", 0.0) + time.time() - t_c
                if crcs:
                    checks[t] = crcs
                shutil.rmtree(os.path.join(raw, f"m{m:02d}"), ignore_errors=True)
                for k in ("clear", "out_of_range", "unstorable", "hazy"):
                    TOTALS["px_" + k] = TOTALS.get("px_" + k, 0) + px[k]
                counts["scenes"] += len(used)
                prog.add(len(scenes_of(m)))
                log.info("[%s-%02d] %d scenes, %.0f %% of the aquifer pixels have a value | %s | %s downloaded | free %.0f GB",
                         tag, m, len(used), 100 * px["covered"] / max(1, F["n_roi"]), prog.line(),
                         human(TOTALS.get("bytes", 0)), free_gb())
                if XP is not np:
                    XP.get_default_memory_pool().free_all_blocks()
                gc.collect()
            status = "complete" if writer.months == list(range(1, 13)) else ("test" if TEST_MODE else "partial")
            writer.commit(status)
            writer.close()
            if old is not None:
                old.close()
                old = None
        except BaseException:
            stop.set()                           # the month being fetched in the background stops too
            pool.shutdown(wait=False, cancel_futures=True)
            raise
        pool.shutdown(wait=True)
        # ---------- the closed file is checked before it may reach Drive ----------
        t_v = time.time()
        compared = sum(readback_month(local, t, crcs) for t, crcs in checks.items())
        good, problems, info = audit_nc(local, mask)
        if not good:
            raise DataIntegrityError(f"{name} failed its audit and was not uploaded: {problems}")
        TIMES["checks"] = TIMES.get("checks", 0.0) + time.time() - t_v
        save_to_drive(local, drive_nc, name)
        size = os.path.getsize(local)
        entry = dict(file=name, tile=key, year=year, months=info["months"], size=size, settings=SIG_ID,
                     frame=frame_id(F), status=info["status"], scenes=counts["scenes"], failed=counts["failed"],
                     header_bytes=header, seconds=round(time.time() - t_start))
        log_events(LEDGER_NAME, [entry])
        LAST.update(file=name)
        TOTALS["files"] = TOTALS.get("files", 0) + 1
        TOTALS["file_bytes"] = TOTALS.get("file_bytes", 0) + size
        log.info("[%s] DONE: months %s, %d scenes read, %s on Drive (%d arrays read back and compared), %.1f min", tag,
                 span(info["months"]), counts["scenes"], human(size), compared, (time.time() - t_start) / 60)
        return dict(entry, action=plan["action"], n_roi=F["n_roi"], new_months=len(months))
    finally:
        for f in (writer, old):
            try:
                if f is not None:
                    f.close()
            except Exception:
                pass
        shutil.rmtree(raw, ignore_errors=True)
        shutil.rmtree(work, ignore_errors=True)


def preflight(units):
    """Fail fast before any bulk work: Drive is writable, the disk floor holds, the engine is
    right, and the local maths reproduces eemont for every satellite."""
    local, remote = os.path.join(NC_DIR, "_write_test.txt"), os.path.join(DRIVE_OUT, "_write_test.txt")
    with open(local, "w") as f:
        f.write(iso(utcnow()))
    robust_drive_copy(local, remote)
    os.remove(remote)
    os.remove(local)
    ensure_disk_free("preflight")
    engine_self_test()
    checks = []
    for sat in SATELLITES:
        cand = [r for year, P in units for m in wanted_months(year) for r in P["months"].get((year, m), [])
                if r["sat"] == sat and r["id"] not in SKIP_SCENES]
        if not cand:
            continue
        cand.sort(key=lambda r: (r["cloud"] < 0, r["cloud"]))
        for r in cand[:3]:                       # the clearest scenes of this satellite in this run
            try:
                res = eemont_cross_check(r)
            except SceneUnavailable:
                res = None
            if res:
                checks.append(res)
                break
    if not checks:
        raise PipelineStop("PREFLIGHT FAILED: no scene could be compared with eemont on Earth Engine.")
    log_events("eemont_cross_check.jsonl", checks)
    log.info("PREFLIGHT OK (%d satellite(s) checked against eemont)", len(checks))


def record_counts():
    """Scenes of the whole job (START_MONTH to STOP_MONTH) over the aquifer's outline, per satellite
    (for the projection)."""
    region, months = ee.Geometry(json.loads(ROI_REGION)), range_months()
    if not months:
        return None
    t0, t1 = iso(month_bounds(*months[0])[0])[:10], iso(month_bounds(*months[-1])[1])[:10]
    try:
        return {s: int(ee.ImageCollection(collection_id(s)).filterBounds(region)
                       .filterDate(t0, t1).size().getInfo()) for s in SATELLITES}
    except Exception as e:
        log.warning("Could not count the whole record: %s", e)
        return None


def run_pipeline():
    t0 = time.time()
    TIMES.clear()
    TOTALS.clear()
    LAST.clear()
    log.info("=" * 78)
    log.info("  Landsat C2 L2 indices, 30 m, monthly medians on tiles | %s | %d indices | TEST_MODE=%s",
             "+".join(SATELLITES), len(INDICES), TEST_MODE)
    log.info("  engine %s | output: %s", ENGINE, DRIVE_OUT)
    log.info("=" * 78)
    results, stopped, prog, tiles = [], None, None, []
    try:
        for d in (RAW_DIR, NC_DIR):              # scratch left by an earlier run is never used
            shutil.rmtree(d, ignore_errors=True)
            os.makedirs(d, exist_ok=True)
        job = job_months(refresh=True)
        if not job:
            raise PipelineStop(f"No month from START_MONTH ({START_MONTH}) to STOP_MONTH ({STOP_MONTH}) is ready yet: "
                               f"a month is ready {SETTLE_DAYS} days after it ended.")
        if TEST_MODE:
            log.info("Rehearsal: months %s of %d (START_MONTH and STOP_MONTH are not used)",
                     span([m for _, m in job]), TEST_YEAR)
        else:
            log.info("Months of this run: %s to %s (%d months)%s", ym(*job[0]), ym(*job[-1]), len(job),
                     "" if LAST_MONTH is None or job[-1] == LAST_MONTH else
                     f" - the months after it up to STOP_MONTH {STOP_MONTH} are not ready yet")
        ee_login()
        load_scale_params()
        ensure_disk_free("start")
        tiles = build_tiles(get_manifest())
        log.info("%d tiles hold aquifer pixels: %s pixels of 30 m in all (UTM zones %s)", len(tiles),
                 f"{sum(P['F']['n_roi'] for P in tiles):,}", ", ".join(sorted({P["key"][1:3] for P in tiles})))
        if ONLY_TILES:
            wanted = set(ONLY_TILES)
            absent = wanted - {P["key"] for P in tiles}
            if absent:
                log.warning("ONLY_TILES not available: %s", sorted(absent))
            tiles = [P for P in tiles if P["key"] in wanted]
        if TEST_MODE and tiles:
            n_pick = min(TEST_TILES, len(tiles))
            tiles = [tiles[int((i + 0.5) * len(tiles) / n_pick)] for i in range(n_pick)]
        units = [(year, P) for year in year_list() for P in tiles if wanted_months(year)]
        if not units:
            raise PipelineStop("Nothing to process - check SATELLITES, the years and ONLY_TILES.")
        n_scenes = lambda year, P: sum(len(P["months"].get((year, m), [])) for m in wanted_months(year))
        prog = Progress(sum(n_scenes(y, P) for y, P in units))
        adopt_old_layout()
        ledger = load_ledger()
        on_drive = sum(os.path.exists(os.path.join(unit_dir(P["key"], y), unit_name(P["key"], y))) for y, P in units)
        log.info("%d yearly files to have (%d tiles, %s to %s, %d scene pieces); %d of them are on Drive already",
                 len(units), len(tiles), ym(*job[0]), ym(*job[-1]), prog.total, on_drive)
        preflight(units)
        skipped = 0
        for k, (year, P) in enumerate(units, 1):
            plan = unit_plan(P, year, ledger)
            if plan["action"] == "skip":
                skipped += 1
                prog.add(n_scenes(year, P), resumed=True)
                continue
            if skipped:
                log.info("%d file(s) already complete on Drive - skipped", skipped)
                skipped = 0
            log.info("-" * 78)
            note = {"new": "", "append": f", added to the months {span(plan['have'])} already in the file",
                    "merge": f", written into a new file with the months {span(plan['have'])} copied from the one on "
                             "Drive"}[plan["action"]]
            log.info("FILE %d/%d  tile %s, %d  (%s aquifer pixels, months %s%s)", k, len(units), P["key"], year,
                     f"{P['F']['n_roi']:,}", span(plan["todo"]), note)
            prog.add(sum(len(P["months"].get((year, m), [])) for m in wanted_months(year) if m in plan["have"]),
                     resumed=True)               # the months of this file that are on Drive already
            results.append(process_unit(P, year, plan, prog))
        if skipped:
            log.info("%d file(s) already complete on Drive - skipped", skipped)
    except PipelineStop as e:
        stopped = str(e)
    except KeyboardInterrupt:
        stopped = "Stopped with the stop button."

    log.info("=" * 78)
    if stopped is not None:
        where = (f"\n  The last file that reached Drive is {LAST['file']}; the next run continues with the one after it."
                 if LAST.get("file") else "\n  No file was finished in this run; the next run starts where this one did.")
        print("\n" + "=" * 78 + f"\n  STOPPED (nothing is lost, re-run to resume)\n  {stopped}{where}\n  Give Drive a "
              "minute to finish uploading before you disconnect the runtime.\n" + "=" * 78 + "\n")
    log.info("  this run: %d yearly file(s) written (%s), %s downloaded in %d blocks, %.1f min", TOTALS.get("files", 0),
             human(TOTALS.get("file_bytes", 0)), human(TOTALS.get("bytes", 0)), TOTALS.get("blocks", 0),
             (time.time() - t0) / 60)
    if TOTALS.get("px_clear"):
        log.info("  usable observations: %s | left out for the valid-range rule: %.3f %% | left out as hazy: %s | "
                 "monthly values a scale could not hold: %d", f"{TOTALS['px_clear']:,}", 100 * TOTALS.get("px_out_of_range", 0)
                 / (TOTALS["px_clear"] + TOTALS.get("px_out_of_range", 0)), f"{TOTALS.get('px_hazy', 0):,}",
                 TOTALS.get("px_unstorable", 0))
    if TIMES:
        total_t = sum(TIMES.values())
        log.info("  time by stage: %s", ", ".join(f"{k} {v:.0f} s ({100 * v / total_t:.0f} %)"
                                                   for k, v in sorted(TIMES.items(), key=lambda kv: -kv[1])))
    fresh = [r for r in results if r.get("header_bytes") and r.get("scenes")]
    if TEST_MODE and prog is not None and prog.run and fresh and not TEST_FRAME_PX:
        total = record_counts()
        if total:
            n_all = sum(total.values())
            per_scene = (TIMES.get("download", 0.0) + TIMES.get("index maths + write", 0.0)) / prog.run
            per_px_month = sum(r["size"] - r["header_bytes"] for r in fresh) / sum(r["n_roi"] * r["new_months"] for r in fresh)
            with open(os.path.join(DRIVE_OUT, FRAMES_NAME)) as f:
                all_px = sum(v["n_roi"] for v in json.load(f)["frames"].values() if v)
            whole = range_months()
            n_months = len(whole)
            log.info("  PROJECTION for the months %s to %s (from this rehearsal, rough; summer months are clearer "
                     "than average):", ym(*whole[0]), ym(*whole[-1]))
            log.info("     %d scenes: %s", n_all, ", ".join(f"{SAT_NAME[s]} {n}" for s, n in total.items()))
            log.info("     ~%.0f h of runtime on this runtime type: %.1f s per scene (%s)", n_all * per_scene / 3600,
                     per_scene, ENGINE.split(" (")[0])
            log.info("     ~%.1f TB to download: %s per scene", n_all * TOTALS.get("bytes", 0) / prog.run / 1024 ** 4,
                     human(TOTALS.get("bytes", 0) / prog.run))
            log.info("     ~%.1f TB on Drive: %.1f bytes per aquifer pixel and month, %s pixels x %d months",
                     all_px * n_months * per_px_month / 1024 ** 4, per_px_month, f"{all_px:,}", n_months)
    log.info("=" * 78)
    return results


RESULTS = run_pipeline()

In [ ]:
# ═══════════════ CELL 10: what is on Drive ═══════════════
AUDIT_OPEN_ALL = TEST_MODE     # True -> open and audit every file (slow for a full output folder);
                               # False -> compare each file's size with the note taken when it was uploaded


def audit_all():
    ledger, by_year = load_ledger(), {}
    for p in glob.glob(os.path.join(DRIVE_OUT, "[0-9][0-9][0-9][0-9]", "z*e*n*", f"{PRODUCT_TAG}_*_{ROI_LABEL}.nc")):
        key, year = os.path.basename(os.path.dirname(p)), os.path.basename(os.path.dirname(os.path.dirname(p)))
        if TILE_RE.fullmatch(key) and os.path.basename(p) == unit_name(key, year):
            by_year.setdefault(int(year), []).append((key, p))
    print(f"{DRIVE_OUT}\n")
    print(f"{'year':4s} {'tiles':>5s} {'with all 12 months':>18s} {'size':>10s}  months held (number of tiles)")
    total, n_files, bad, notes = 0, 0, 0, []
    for year in sorted(by_year):
        size, whole, held = 0, 0, {}
        for key, p in sorted(by_year[year]):
            name = os.path.basename(p)
            led, sz = ledger.get(name), os.path.getsize(p)
            size += sz
            if AUDIT_OPEN_ALL or led is None or led.get("size") != sz:
                ok, problems, info = audit_nc(p)
                months = info.get("months", [])
                if not ok:
                    bad += 1
                    notes.append(f"{year} {key}: " + "; ".join(problems))
                elif led is None or led.get("size") != sz:
                    notes.append(f"{year} {key}: not as noted at upload, but it passes the audit")
            else:
                months = [int(m) for m in led.get("months", [])]
            whole += months == list(range(1, 13))
            held[span(months)] = held.get(span(months), 0) + 1
        n_files += len(by_year[year])
        total += size
        print(f"{year:4d} {len(by_year[year]):5d} {whole:18d} {human(size):>10s}  "
              + ", ".join(f"{s or 'none'} ({n})" for s, n in sorted(held.items(), key=lambda kv: -kv[1])))
    print(f"\n{n_files} files, {human(total)} | {bad} file(s) with problems")
    for note in notes:
        print("  " + note)
    earlier = [p for d in glob.glob(os.path.join(DRIVE_OUT, "z*e*n*")) if os.path.isdir(d)
               for p in glob.glob(os.path.join(d, "*.nc"))]
    if earlier:
        print(f"{len(earlier)} file(s) are still in the earlier folder layout <tile>/ ; the next run of Cell 9 moves them")
    for name in (LOG_FAILED, LOG_SKIPPED):
        path, seen = os.path.join(DRIVE_OUT, name), set()
        if os.path.exists(path):
            for line in open(path):
                try:
                    seen.add(json.loads(line).get("scene"))
                except ValueError:
                    pass
        print(f"{name}: {len(seen)} scenes")


audit_all()

# Where the files are: <output folder>/<year>/<tile>/<file>.nc, one file per tile and year.
#
# Reading them. Values are 16-bit integers with a scale factor; xarray turns them into floats and the
# fill value into NaN by itself:
#   import glob, xarray as xr
#   root  = "/content/drive/MyDrive/MSUGWB/LANDSAT_C2L2_indices_30m_monthly_tiles_v2"
#   files = sorted(glob.glob(f"{root}/*/z14e02n29/*.nc"))                          # every year of one tile
#   ds = xr.open_mfdataset(files, combine="nested", concat_dim="time", data_vars="minimal", coords="minimal",
#                          compat="override", chunks={"time": 1, "y": 2048, "x": 2048})     # lazily
#   ndvi = ds.NDVI.where(ds.n_obs >= 2)            # e.g. keep months with at least two usable observations
# Every yearly file of a tile has the same x and y, so the years stack without any regridding.
# sorted(glob.glob(f"{root}/2005/z14*/*.nc")) lists the tiles of UTM zone 14 for one year. Tiles of one
# zone are pieces of one 30 m grid: their x and y values line up exactly, with no regridding needed.
#
# Optional independent CF check of one file:
# !pip -q install compliance-checker
# !compliance-checker --test=cf:1.9 "<file>.nc"